In [ ]:
# Configure SCBFM_ROOT_DIR and optionally SCBFM_FIGURE_DIR before launching Jupyter.
from pathlib import Path
import os
import sys

_candidates = [Path(os.environ['SCBFM_REPO_DIR'])] if os.environ.get('SCBFM_REPO_DIR') else []
_candidates += [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p / 'src' / 'main.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the checkout or set SCBFM_REPO_DIR.')
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from notebook_setup import ROOT_DIR, OUTPUT_DIR, FIGURE_DIR


In [ ]:
import pandas as pd
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np  # likely needed too

import json
import glob
import os

# Set as default
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

HEAD_ONLY_BACKBONE_EVAL_TASKS = {
    'canc_type_class', 'canc_type_class_33', 'deconv', 'disease_class',
    'drug_resp', 'gene_essent', 'surv_pred', 'surv_pred_binary',
    'surv_pred_survboard',
}


def head_only_output_is_current(base, task, mode):
    if task not in HEAD_ONLY_BACKBONE_EVAL_TASKS or mode == 'pca_rf':
        return True
    metadata_path = f'{base}/{mode}/{task}_{mode}_run_metadata.json'
    metrics_path = f'{base}/{mode}/{task}_{mode}_evaluation_metrics.csv'
    if os.path.exists(metadata_path) and os.path.exists(metrics_path):
        with open(metadata_path) as handle:
            metadata = json.load(handle)
        valid_head_only = (
            mode != 'head_only' or metadata.get('head_only_backbone_eval') is True
        )
        if metadata.get('run_status') == 'complete' and valid_head_only:
            return True
    print(f'Skipping incomplete or incompatible output for {task}/{mode}')
    return False


def completed_output_is_current(base, task, variant):
    metadata_path = f'{base}/{variant}/{task}_{variant}_run_metadata.json'
    if not os.path.exists(metadata_path):
        return False
    with open(metadata_path) as handle:
        metadata = json.load(handle)
    return metadata.get('run_status') == 'complete'

FINETUNING_CURVE_MODELS = [
    "random_init", "pretrain_sc", "preadapt_sc",
    "pretrain_bulk", "preadapt_bulk",
]
FINETUNING_CURVE_COLORS = {
    "random_init": "#4C72B0", "pretrain_sc": "#55A868",
    "preadapt_sc": "#2C7FB8", "pretrain_bulk": "#DD8452",
    "preadapt_bulk": "#C44E52", "raw_mlp": "#666666",
}


def plot_finetuning_convergence(
    task,
    title,
    loss_label,
    *,
    nested_by_cohort=False,
    extra_mlp_variants=(),
):
    base = str(OUTPUT_DIR / task)
    panel_specs = [
        ("head_only", "Head", FINETUNING_CURVE_MODELS),
        ("adapters", "Adapters", FINETUNING_CURVE_MODELS),
        ("full_ft", "Full", FINETUNING_CURVE_MODELS),
        ("raw_mlp_all_genes", "RE MLP, All", ["raw_mlp"]),
        ("raw_mlp_hvg1199", "RE MLP, MAD", ["raw_mlp"]),
        *[(variant, label, ["raw_mlp"]) for variant, label in extra_mlp_variants],
    ]

    available_panels = []
    for variant, panel_title, models in panel_specs:
        series = {}
        for model in models:
            filename = f"{task}_{variant}_{model}_training_curves.csv"
            if nested_by_cohort:
                paths = sorted(glob.glob(f"{base}/*/{variant}/{filename}"))
            else:
                path = f"{base}/{variant}/{filename}"
                paths = [path] if os.path.exists(path) else []
            frames = []
            for path in paths:
                frame = pd.read_csv(path, comment="#")
                if not {"epoch", "validation_loss"}.issubset(frame.columns):
                    continue
                frame = frame[["epoch", "validation_loss"]].copy()
                frame["epoch"] = pd.to_numeric(frame["epoch"], errors="coerce")
                frame["validation_loss"] = pd.to_numeric(
                    frame["validation_loss"], errors="coerce"
                )
                frame = frame.dropna(subset=["epoch", "validation_loss"])
                if not frame.empty:
                    frames.append(frame)
            if frames:
                series[model] = pd.concat(frames, ignore_index=True)
        if series:
            available_panels.append((variant, panel_title, series))

    if not available_panels:
        raise FileNotFoundError(f"No training curves found under {base}")

    n_columns = 2
    n_rows = int(np.ceil(len(available_panels) / n_columns))
    fig, axes = plt.subplots(
        n_rows, n_columns, figsize=(12, 4.0 * n_rows), squeeze=False
    )
    for ax, (_variant, panel_title, series) in zip(axes.flat, available_panels):
        positive_loss = True
        summaries = {}
        for model, frame in series.items():
            summary = (
                frame.groupby("epoch")["validation_loss"]
                .agg(["mean", "std"])
                .reset_index()
            )
            summaries[model] = summary
            positive_loss = positive_loss and bool((summary["mean"] > 0).all())
        for model, summary in summaries.items():
            epochs = summary["epoch"].to_numpy()
            means = summary["mean"].to_numpy()
            stds = summary["std"].fillna(0.0).to_numpy()
            color = FINETUNING_CURVE_COLORS.get(model, "#666666")
            ax.plot(
                epochs, means, marker="o", markersize=3, lw=1.8,
                color=color, label=model.replace("_", " "),
            )
            lower = means - stds
            if positive_loss:
                lower = np.maximum(lower, np.finfo(float).tiny)
            ax.fill_between(epochs, lower, means + stds, color=color, alpha=0.12)
        if positive_loss:
            ax.set_yscale("log")
        ax.set_title(panel_title, fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(loss_label)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(fontsize=8, framealpha=0.9)
    for ax in axes.flat[len(available_panels):]:
        ax.set_visible(False)
    scope = "folds and cohorts" if nested_by_cohort else "folds"
    fig.suptitle(f"{title} convergence across {scope}", fontsize=13)
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()
    return None


print("Fonts loaded, good to go!")


## Pretraining

In [ ]:
root = str(OUTPUT_DIR)

def load_val(path):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    val = df[df["split"] == "val"].reset_index(drop=True)
    return val if not val.empty else None

series = {
    "scRNA-seq": {
        "pretrain": load_val(f"{root}/pretrain_sc/pretrain_sc_pretrain_epoch_metrics.csv"),
        "pa0": load_val(f"{root}/preadapt_sc/preadapt_sc_pa0_epoch_metrics.csv"),
        "preadapt": load_val(f"{root}/preadapt_sc/preadapt_sc_pretrain_epoch_metrics.csv"),
        "color": "#356B9A",
    },
    "bulkRNA-seq": {
        "pretrain": load_val(f"{root}/pretrain_bulk/pretrain_bulk_pretrain_epoch_metrics.csv"),
        "pa0": load_val(f"{root}/preadapt_bulk/preadapt_bulk_pa0_epoch_metrics.csv"),
        "preadapt": load_val(f"{root}/preadapt_bulk/preadapt_bulk_pretrain_epoch_metrics.csv"),
        "color": "#C46A3A",
    },
}
series = {name: values for name, values in series.items()
          if any(values[key] is not None for key in ("pretrain", "pa0", "preadapt"))}
if not series:
    raise FileNotFoundError(f"No pre-training, PA0, or pre-adaptation metric files found under {root}")

pretrain_epochs = sorted({epoch for values in series.values()
                          if values["pretrain"] is not None
                          for epoch in values["pretrain"]["epoch"]})
pretrain_end = max(pretrain_epochs, default=0)
has_pa0 = any(values["pa0"] is not None for values in series.values())
pa0_tick = pretrain_end + 1 if has_pa0 else None
preadapt_epochs = sorted({epoch for values in series.values()
                          if values["preadapt"] is not None
                          for epoch in values["preadapt"]["epoch"]})
preadapt_offset = pretrain_end + (1 if has_pa0 else 0)
preadapt_ticks = [preadapt_offset + epoch for epoch in preadapt_epochs]
sep = pretrain_end + 0.5

fig, ax = plt.subplots(figsize=(7.2, 3.6), layout="constrained")

for label, values in series.items():
    df_pt = values["pretrain"]
    df_pa0 = values["pa0"]
    df_pa = values["preadapt"]
    color = values["color"]
    pretrain_end_x = None
    pretrain_end_y = None
    endpoint_x = None
    endpoint_y = None

    if df_pt is not None:
        ax.plot(
            df_pt["epoch"], df_pt["loss"], color=color, marker="o",
            markersize=3.8, markeredgewidth=0, lw=1.65, zorder=3,
        )
        pretrain_end_x = float(df_pt["epoch"].iloc[-1])
        pretrain_end_y = float(df_pt["loss"].iloc[-1])
        endpoint_x = pretrain_end_x
        endpoint_y = pretrain_end_y

    pa_x = []
    pa_y = []
    if df_pa0 is not None and pa0_tick is not None:
        pa0_loss = float(df_pa0["loss"].iloc[0])
        if pretrain_end_x is not None:
            ax.plot(
                [pretrain_end_x, pa0_tick], [pretrain_end_y, pa0_loss],
                color=color, lw=1.2, ls=(0, (3, 2)), alpha=0.75, zorder=2,
            )
        pa_x.append(float(pa0_tick))
        pa_y.append(pa0_loss)

    if df_pa is not None:
        pa_x.extend((preadapt_offset + df_pa["epoch"].to_numpy()).tolist())
        pa_y.extend(df_pa["loss"].astype(float).tolist())

    if pa_x:
        ax.plot(
            pa_x, pa_y, color=color, marker="o", markersize=3.8,
            markeredgewidth=0, lw=1.65, zorder=3,
        )
        endpoint_x = pa_x[-1]
        endpoint_y = pa_y[-1]

    if endpoint_x is not None:
        y_offset = 5 if label == "scRNA-seq" else -8
        ax.annotate(
            label, xy=(endpoint_x, endpoint_y), xytext=(7, y_offset),
            textcoords="offset points", color=color, fontsize=8.5,
            ha="left", va="center", annotation_clip=False,
        )

if pretrain_epochs and (has_pa0 or preadapt_epochs):
    ax.axvline(sep, color="#B8B8B8", lw=0.8, zorder=1)
if pretrain_epochs:
    ax.text(
        np.mean(pretrain_epochs), 1.035, "Pretraining",
        transform=ax.get_xaxis_transform(), ha="center", va="bottom",
        fontsize=9, color="#4A4A4A", clip_on=False,
    )
pa_phase_ticks = ([pa0_tick] if pa0_tick is not None else []) + preadapt_ticks
if pa_phase_ticks:
    ax.text(
        np.mean(pa_phase_ticks), 1.035, "Pre-adaptation",
        transform=ax.get_xaxis_transform(), ha="center", va="bottom",
        fontsize=9, color="#4A4A4A", clip_on=False,
    )

xticks = pretrain_epochs + ([pa0_tick] if pa0_tick is not None else []) + preadapt_ticks
xlabels = (
    [str(int(epoch)) for epoch in pretrain_epochs]
    + (["PA0"] if pa0_tick is not None else [])
    + [f"PA{int(epoch)}" for epoch in preadapt_epochs]
)
ax.set_yscale("log")
ax.set_ylim(125, 850)
ax.set_yticks([150, 200, 300, 500, 800])
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())
ax.set_xticks(xticks)
ax.set_xticklabels(xlabels)
ax.set_xlim(min(xticks) - 0.25, max(xticks) + 1.7)
ax.set_xlabel("Epoch", fontsize=9.5, labelpad=7)
ax.grid(axis="y", which="major", color="#E5E5E5", lw=0.65)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#555555")
    ax.spines[side].set_linewidth(0.8)
ax.tick_params(axis="both", which="major", labelsize=8.5, length=3,
               width=0.8, color="#555555")

figure_dir = str(FIGURE_DIR)
os.makedirs(figure_dir, exist_ok=True)
fig.savefig(os.path.join(figure_dir, "pretrain.pdf"), bbox_inches="tight")
fig.savefig(
    os.path.join(figure_dir, "pretrain.png"), dpi=300,
    bbox_inches="tight", facecolor="white",
)
plt.show()


### scGPT pre-adaptation

In [ ]:
scgpt_preadapt_path = f"{root}/scgpt_preadapt/scgpt_preadapt_epoch_metrics.csv"
if not os.path.exists(scgpt_preadapt_path):
    raise FileNotFoundError(f"No scGPT pre-adaptation metrics found at {scgpt_preadapt_path}")

scgpt_preadapt_metrics = pd.read_csv(scgpt_preadapt_path)
required_columns = {"epoch", "split", "scope", "loss"}
missing_columns = required_columns.difference(scgpt_preadapt_metrics.columns)
if missing_columns:
    raise ValueError(f"scGPT pre-adaptation metrics are missing columns: {sorted(missing_columns)}")

scgpt_validation = (
    scgpt_preadapt_metrics[
        (scgpt_preadapt_metrics["split"] == "validation")
        & (scgpt_preadapt_metrics["scope"] == "all")
    ]
    .sort_values("epoch")
    .reset_index(drop=True)
)
if scgpt_validation.empty:
    raise ValueError("No overall validation rows found in the scGPT pre-adaptation metrics.")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(
    scgpt_validation["epoch"],
    scgpt_validation["loss"],
    color="#4C72B0",
    marker="o",
    lw=2,
)
epochs = scgpt_validation["epoch"].astype(int).tolist()
ax.set_yscale("log")
ax.set_xticks(epochs)
ax.set_xlabel("Pre-adaptation epoch", fontsize=11)
ax.set_ylabel("Validation loss (log scale)", fontsize=11)
ax.set_title("scGPT pre-adaptation - Validation Loss", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


### Bulk pretraining dataset-size comparison

In [ ]:
bulk_scale_specs = [
    ("10k", ("pretrain_bulk_10k",)),
    ("50k", ("pretrain_bulk_50k",)),
    ("100k", ("pretrain_bulk_100k",)),
    ("200k", ("pretrain_bulk_200k",)),
    ("400k", ("pretrain_bulk_400k",)),
    ("Full", ("pretrain_bulk",)),
]

bulk_scale_series = {}
for label, model_names in bulk_scale_specs:
    for model_name in model_names:
        path = f"{root}/{model_name}/{model_name}_pretrain_epoch_metrics.csv"
        values = load_val(path)
        if values is not None:
            bulk_scale_series[label] = values
            break

if not bulk_scale_series:
    raise FileNotFoundError(f"No bulk pretraining scale metrics found under {root}")

colors = plt.get_cmap("viridis")(
    np.linspace(0.08, 0.92, len(bulk_scale_series))
)
fig, ax = plt.subplots(figsize=(8, 4.5))
for (label, values), color in zip(bulk_scale_series.items(), colors):
    ax.plot(
        values["epoch"],
        values["loss"],
        color=color,
        marker="o",
        lw=2,
        label=label,
    )

epochs = sorted({
    int(epoch)
    for values in bulk_scale_series.values()
    for epoch in values["epoch"]
})
ax.set_yscale("log")
ax.set_xticks(epochs)
ax.set_xlabel("Pretraining epoch", fontsize=11)
ax.set_ylabel("Validation loss (log scale)", fontsize=11)
ax.set_title("Bulk pretraining by dataset size", fontsize=12)
ax.legend(title="Selected samples", ncol=2, fontsize=9, framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


## Cancer classification (5 types)

In [ ]:
bulkrnabert_weighted_f1_mean = 0.993
bulkrnabert_weighted_f1_sd = 0.004


In [ ]:
task = "canc_type_class"
base = str(OUTPUT_DIR / task)
modes = ["head_only", "adapters", "full_ft", "pca_rf"]
mode_labels = {"head_only": "Head", "adapters": "Adapters", "full_ft": "Full", "pca_rf": "PCA+RF"}
colors  = {"head_only": "#4C72B0", "adapters": "#DD8452", "full_ft": "#55A868", "pca_rf": "#8172B3"}
model_keys = ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]
metric_mean = "f1_weighted_mean"
metric_std = "f1_weighted_std"

baseline_specs = [
    ("raw_mlp_all_genes", "RE MLP\nAll"),
    ("raw_mlp_hvg1199", "RE MLP\nMAD"),
    ("raw_pca_rf_all_genes", "RE PCA+RF\nAll"),
    ("raw_pca_rf_hvg1199", "RE PCA+RF\nMAD"),
]
large_fm_specs = [
    ("scgpt_pca_rf", "scGPT"),
    ("scgpt_preadapt_pca_rf", "Preadapted\nscGPT"),
    ("bulkformer_pca_rf", "BulkFormer"),
]

def load_eval_csv(path):
    if os.path.exists(path):
        return pd.read_csv(path, comment="#").set_index("model")
    return None


def load_mode(mode):
    if not head_only_output_is_current(base, task, mode):
        return None
    combined = f"{base}/{mode}/{task}_{mode}_evaluation_metrics.csv"
    df = load_eval_csv(combined)
    if df is not None:
        return df
    parts = [
        pd.read_csv(f"{base}/{mode}/{task}_{mode}_{m}_evaluation_metrics.csv", comment="#")
        for m in model_keys
        if os.path.exists(f"{base}/{mode}/{task}_{mode}_{m}_evaluation_metrics.csv")
    ]
    return pd.concat(parts).set_index("model") if parts else None


def load_variant_metrics(specs, metric_col):
    loaded = {}
    for spec in specs:
        canonical = spec[0]
        if not completed_output_is_current(base, task, canonical):
            print(f'Skipping stale or partial result: {canonical}')
            continue
        path = f"{base}/{canonical}/{task}_{canonical}_evaluation_metrics.csv"
        df = load_eval_csv(path)
        if df is not None and metric_col in df.columns:
            loaded[canonical] = df
    return loaded


dfs = {mode: load_mode(mode) for mode in modes}
available_modes = [mode for mode in modes if dfs[mode] is not None]
baseline_dfs = load_variant_metrics(baseline_specs, metric_mean)
large_fm_dfs = load_variant_metrics(large_fm_specs, metric_mean)
available_baselines = [variant for variant, _ in baseline_specs if variant in baseline_dfs]
available_large_fms = [variant for variant, _label in large_fm_specs if variant in large_fm_dfs]
if not available_modes and not available_baselines and not available_large_fms:
    raise FileNotFoundError(f"No evaluation metric files found under {base}")

bar_w = 0.18
centers = {
    "random_init":  0.0,
    "pretrain_sc":  1.1,
    "preadapt_sc":  1.9,
    "pretrain_bulk": 3.0,
    "preadapt_bulk": 3.8,
}
baseline_start = 5.0
baseline_spacing = 0.78
baseline_x = {variant: baseline_start + i * baseline_spacing for i, variant in enumerate(available_baselines)}
large_fm_start = (max(baseline_x.values()) + 0.95) if baseline_x else 5.0
large_fm_spacing = 0.78
large_fm_x = {variant: large_fm_start + i * large_fm_spacing for i, variant in enumerate(available_large_fms)}
rightmost_nonref = max(
    list(baseline_x.values())
    + list(large_fm_x.values())
    + [centers["preadapt_bulk"]]
)
brnabert_x = rightmost_nonref + 1.05

fig, ax = plt.subplots(figsize=(17, 5.5))

added = set()
for m_idx, mode in enumerate(available_modes):
    offset = (m_idx - (len(available_modes) - 1) / 2) * bar_w
    for model, xc in centers.items():
        df = dfs[mode]
        if df is None or model not in df.index or metric_mean not in df.columns:
            continue
        mean = df.loc[model, metric_mean]
        std = df.loc[model, metric_std] if metric_std in df.columns else 0.0
        lbl = mode_labels[mode] if mode not in added else ""
        ax.bar(xc + offset, mean, bar_w, yerr=std, color=colors[mode],
               capsize=3, alpha=0.88, label=lbl, error_kw={"linewidth": 1.2})
        added.add(mode)

for variant, label in baseline_specs:
    if variant not in baseline_dfs:
        continue
    row = baseline_dfs[variant].loc["raw_mlp"] if "raw_mlp" in baseline_dfs[variant].index else baseline_dfs[variant].iloc[0]
    mean = row[metric_mean]
    std = row[metric_std] if metric_std in row.index else 0.0
    ax.bar(baseline_x[variant], mean, bar_w * 1.2, yerr=std, capsize=3,
           color="#8C8C8C", alpha=0.88,
           label="Baselines" if variant == available_baselines[0] else "",
           error_kw={"linewidth": 1.2})

for variant, label in large_fm_specs:
    if variant not in large_fm_dfs or variant not in large_fm_x:
        continue
    row = large_fm_dfs[variant].loc["raw_mlp"] if "raw_mlp" in large_fm_dfs[variant].index else large_fm_dfs[variant].iloc[0]
    mean = row[metric_mean]
    std = row[metric_std] if metric_std in row.index else 0.0
    ax.bar(large_fm_x[variant], mean, bar_w * 1.2, yerr=std, capsize=3,
           color="#8172B3", alpha=0.88,
           label="_nolegend_",
           error_kw={"linewidth": 1.2})

ax.bar(brnabert_x, bulkrnabert_weighted_f1_mean, bar_w * 1.5,
       yerr=bulkrnabert_weighted_f1_sd, capsize=3,
       color="#999999", alpha=0.88, label="_nolegend_")

sc_lo, sc_hi = centers["pretrain_sc"] - 0.45, centers["preadapt_sc"] + 0.45
bulk_lo, bulk_hi = centers["pretrain_bulk"] - 0.45, centers["preadapt_bulk"] + 0.45
baseline_lo = (min(baseline_x.values()) - 0.42) if baseline_x else None
baseline_hi = (max(baseline_x.values()) + 0.42) if baseline_x else None
large_fm_lo = (min(large_fm_x.values()) - 0.42) if large_fm_x else None
large_fm_hi = (max(large_fm_x.values()) + 0.42) if large_fm_x else None
ref_lo, ref_hi = brnabert_x - 0.42, brnabert_x + 0.42
ax.axvspan(sc_lo, sc_hi, alpha=0.07, color="steelblue", zorder=0)
ax.axvspan(bulk_lo, bulk_hi, alpha=0.07, color="salmon", zorder=0)
if baseline_x:
    ax.axvspan(baseline_lo, baseline_hi, alpha=0.06, color="darkgray", zorder=0)
if large_fm_x:
    ax.axvspan(large_fm_lo, large_fm_hi, alpha=0.07, color="#8172B3", zorder=0)
ax.axvspan(ref_lo, ref_hi, alpha=0.05, color="gray", zorder=0)

y_lim = (0, 1.16)
ax.set_ylim(*y_lim)
ax.text((sc_lo + sc_hi) / 2, y_lim[1] - 0.015, "sc pre-trained", ha="center", va="top", fontsize=9, color="steelblue", style="italic")
ax.text((bulk_lo + bulk_hi) / 2, y_lim[1] - 0.015, "bulk pre-trained", ha="center", va="top", fontsize=9, color="firebrick", style="italic")
if baseline_x:
    ax.text((baseline_lo + baseline_hi) / 2, y_lim[1] - 0.015, "baselines", ha="center", va="top", fontsize=9, color="dimgray", style="italic")
if large_fm_x:
    ax.text((large_fm_lo + large_fm_hi) / 2, y_lim[1] - 0.015, "large FMs", ha="center", va="top", fontsize=9, color="#8172B3", style="italic")
ax.text((ref_lo + ref_hi) / 2, y_lim[1] - 0.015, "reference models", ha="center", va="top", fontsize=9, color="dimgray", style="italic")

large_fm_label_map = {variant: label for variant, label in large_fm_specs}
tick_pos = (
    list(centers.values())
    + list(baseline_x.values())
    + [large_fm_x[v] for v in available_large_fms]
    + [brnabert_x]
)
tick_labels = (
    ["Random\ninit", "Pretrain\nsc", "Preadapt\nsc", "Pretrain\nbulk", "Preadapt\nbulk"]
    + [dict(baseline_specs)[v] for v in available_baselines]
    + [large_fm_label_map[v] for v in available_large_fms]
    + ["BulkRNA-\nBert"]
)
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, fontsize=8.8)
ax.set_ylabel("Weighted F1", fontsize=11)
ax.set_title("Cancer Classification (5 types) — Weighted F1", fontsize=12)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9.5, framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout(rect=(0, 0, 0.86, 1))
plt.show()


### Bulk pretraining dataset-size downstream comparison

In [ ]:
task = "canc_type_class"
base = str(OUTPUT_DIR / task)
metric_mean = "f1_weighted_mean"
metric_std = "f1_weighted_std"

bulk_downstream_specs = [
    ("10k", "head_only_pretrain_bulk_10k"),
    ("50k", "head_only_pretrain_bulk_50k"),
    ("100k", "head_only_pretrain_bulk_100k"),
    ("200k", "head_only_pretrain_bulk_200k"),
    ("400k", "head_only_pretrain_bulk_400k"),
    ("Full", None),
]


def load_bulk_downstream_result(label, variant):
    if label == "Full":
        metadata_path = f"{base}/head_only/{task}_head_only_run_metadata.json"
        candidates = [
            f"{base}/head_only/{task}_head_only_pretrain_bulk_evaluation_metrics.csv",
            f"{base}/head_only/{task}_head_only_evaluation_metrics.csv",
        ]
    else:
        prefix = f"{task}_{variant}"
        metadata_path = f"{base}/{variant}/{prefix}_run_metadata.json"
        candidates = [
            f"{base}/{variant}/{prefix}_evaluation_metrics.csv",
            f"{base}/{variant}/{prefix}_pretrain_bulk_evaluation_metrics.csv",
        ]

    if not os.path.exists(metadata_path):
        return None, None, None
    with open(metadata_path) as handle:
        metadata = __import__("json").load(handle)
    if (
        metadata.get("head_only_backbone_eval") is not True
        or metadata.get("run_status") != "complete"
    ):
        print(f"Skipping incomplete or incompatible head-only result: {label}")
        return None, None, None
    fold_fingerprint = metadata.get("cv_fold_fingerprint")
    if not fold_fingerprint:
        print(f"Skipping head-only result without fold fingerprint: {label}")
        return None, None, None

    for path in candidates:
        if not os.path.exists(path):
            continue
        frame = pd.read_csv(path, comment="#")
        if "model" in frame.columns:
            frame = frame[frame["model"] == "pretrain_bulk"]
        if not frame.empty and metric_mean in frame.columns:
            return frame.iloc[0], path, fold_fingerprint
    return None, None, None


bulk_downstream_rows = []
for label, variant in bulk_downstream_specs:
    row, path, fold_fingerprint = load_bulk_downstream_result(label, variant)
    if row is not None:
        bulk_downstream_rows.append({
            "label": label,
            "mean": float(row[metric_mean]),
            "std": float(row[metric_std]) if metric_std in row.index else 0.0,
            "path": path,
            "fold_fingerprint": fold_fingerprint,
        })

if not bulk_downstream_rows:
    raise FileNotFoundError(f"No complete bulk-size head-only results found under {base}")
if len({row["fold_fingerprint"] for row in bulk_downstream_rows}) != 1:
    raise ValueError("Bulk-size head-only results do not use identical CV folds")

x = np.arange(len(bulk_downstream_rows))
means = np.asarray([row["mean"] for row in bulk_downstream_rows])
stds = np.asarray([row["std"] for row in bulk_downstream_rows])
bar_colors = plt.get_cmap("viridis")(
    np.linspace(0.12, 0.88, len(bulk_downstream_rows))
)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(
    x, means, yerr=stds, capsize=4, width=0.68,
    color=bar_colors, alpha=0.9, error_kw={"linewidth": 1.2},
)
for bar, mean in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2, mean + 0.018, f"{mean:.3f}",
        ha="center", va="bottom", fontsize=9,
    )

ax.set_xticks(x)
ax.set_xticklabels([row["label"] for row in bulk_downstream_rows])
ax.set_ylim(0, 1.08)
ax.set_xlabel("Bulk pretraining samples", fontsize=11)
ax.set_ylabel("Weighted F1", fontsize=11)
ax.set_title("Cancer classification: head-only performance by pretraining size", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


### Validation-loss convergence


In [ ]:
plot_finetuning_convergence(
    "canc_type_class",
    "Cancer classification (5 types)",
    "Validation weighted cross-entropy loss",
)


## Cancer classification (33 types)

In [ ]:
bulkrnabert_weighted_f1_mean = 0.942
bulkrnabert_weighted_f1_sd = 0.004
bulkformer_weighted_f1_mean = 0.833
geneformer_weighted_f1_mean = 0.473
genecompass_weighted_f1_mean = 0.761
scgpt_weighted_f1_mean = 0.83
scfoundation_weighted_f1_mean = 0.791
sclong_weighted_f1_mean = 0.347


In [ ]:
task = "canc_type_class_33"
base = str(OUTPUT_DIR / task)
modes = ["head_only", "adapters", "full_ft", "pca_rf"]
mode_labels = {"head_only": "Head", "adapters": "Adapters", "full_ft": "Full", "pca_rf": "PCA+RF"}
colors  = {"head_only": "#4C72B0", "adapters": "#DD8452", "full_ft": "#55A868", "pca_rf": "#8172B3"}
model_keys = ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]
metric_mean = "f1_weighted_mean"
metric_std = "f1_weighted_std"

baseline_specs = [
    ("raw_mlp_all_genes", "RE MLP\nAll"),
    ("raw_mlp_hvg1199", "RE MLP\nMAD"),
    ("raw_pca_rf_all_genes", "RE PCA+RF\nAll"),
    ("raw_pca_rf_hvg1199", "RE PCA+RF\nMAD"),
]
large_fm_specs = [
    ("scgpt_pca_rf", "scGPT"),
    ("scgpt_preadapt_pca_rf", "Preadapted\nscGPT"),
    ("bulkformer_pca_rf", "BulkFormer"),
]

def load_eval_csv(path):
    if os.path.exists(path):
        return pd.read_csv(path, comment="#").set_index("model")
    return None


def load_mode(mode):
    if not head_only_output_is_current(base, task, mode):
        return None
    combined = f"{base}/{mode}/{task}_{mode}_evaluation_metrics.csv"
    df = load_eval_csv(combined)
    if df is not None:
        return df
    parts = [
        pd.read_csv(f"{base}/{mode}/{task}_{mode}_{m}_evaluation_metrics.csv", comment="#")
        for m in model_keys
        if os.path.exists(f"{base}/{mode}/{task}_{mode}_{m}_evaluation_metrics.csv")
    ]
    return pd.concat(parts).set_index("model") if parts else None


def load_variant_metrics(specs, metric_col):
    loaded = {}
    for spec in specs:
        canonical = spec[0]
        if canonical in {'scgpt_pca_rf', 'scgpt_preadapt_pca_rf', 'bulkformer_pca_rf'} and not completed_output_is_current(base, task, canonical):
            print(f'Skipping stale or partial external baseline: {canonical}')
            continue
        path = f"{base}/{canonical}/{task}_{canonical}_evaluation_metrics.csv"
        df = load_eval_csv(path)
        if df is not None and metric_col in df.columns:
            loaded[canonical] = df
    return loaded


dfs = {mode: load_mode(mode) for mode in modes}
available_modes = [mode for mode in modes if dfs[mode] is not None]
baseline_dfs = load_variant_metrics(baseline_specs, metric_mean)
large_fm_dfs = load_variant_metrics(large_fm_specs, metric_mean)
available_baselines = [variant for variant, _ in baseline_specs if variant in baseline_dfs]
available_large_fms = [variant for variant, _label in large_fm_specs if variant in large_fm_dfs]
if not available_modes and not available_baselines and not available_large_fms:
    raise FileNotFoundError(f"No evaluation metric files found under {base}")

bar_w = 0.18
centers = {
    "random_init":  0.0,
    "pretrain_sc":  1.1,
    "preadapt_sc":  1.9,
    "pretrain_bulk": 3.0,
    "preadapt_bulk": 3.8,
}
baseline_start = 5.0
baseline_spacing = 0.78
baseline_x = {variant: baseline_start + i * baseline_spacing for i, variant in enumerate(available_baselines)}
large_fm_start = (max(baseline_x.values()) + 0.95) if baseline_x else 5.0
large_fm_spacing = 0.78
large_fm_x = {variant: large_fm_start + i * large_fm_spacing for i, variant in enumerate(available_large_fms)}
rightmost_nonref = max(
    list(baseline_x.values())
    + list(large_fm_x.values())
    + [centers["preadapt_bulk"]]
)
ref_models = [
    ("BulkRNA-Bert", bulkrnabert_weighted_f1_mean, bulkrnabert_weighted_f1_sd),
    ("BulkFormer\n(reported)", bulkformer_weighted_f1_mean, None),
    ("GeneFormer", geneformer_weighted_f1_mean, None),
    ("GeneCompass", genecompass_weighted_f1_mean, None),
    ("scGPT\n(reported)", scgpt_weighted_f1_mean, None),
    ("scFoundation", scfoundation_weighted_f1_mean, None),
    ("scLong", sclong_weighted_f1_mean, None),
]
ref_start = rightmost_nonref + 1.05
ref_spacing = 0.78
ref_x = [ref_start + i * ref_spacing for i in range(len(ref_models))]

fig, ax = plt.subplots(figsize=(21, 5.5))

added = set()
for m_idx, mode in enumerate(available_modes):
    offset = (m_idx - (len(available_modes) - 1) / 2) * bar_w
    for model, xc in centers.items():
        df = dfs[mode]
        if df is None or model not in df.index or metric_mean not in df.columns:
            continue
        mean = df.loc[model, metric_mean]
        std = df.loc[model, metric_std] if metric_std in df.columns else 0.0
        lbl = mode_labels[mode] if mode not in added else ""
        ax.bar(xc + offset, mean, bar_w, yerr=std, color=colors[mode],
               capsize=3, alpha=0.88, label=lbl, error_kw={"linewidth": 1.2})
        added.add(mode)

for variant, label in baseline_specs:
    if variant not in baseline_dfs:
        continue
    row = baseline_dfs[variant].loc["raw_mlp"] if "raw_mlp" in baseline_dfs[variant].index else baseline_dfs[variant].iloc[0]
    mean = row[metric_mean]
    std = row[metric_std] if metric_std in row.index else 0.0
    ax.bar(baseline_x[variant], mean, bar_w * 1.2, yerr=std, capsize=3,
           color="#8C8C8C", alpha=0.88,
           label="Baselines" if variant == available_baselines[0] else "",
           error_kw={"linewidth": 1.2})

for variant, label in large_fm_specs:
    if variant not in large_fm_dfs or variant not in large_fm_x:
        continue
    row = large_fm_dfs[variant].loc["raw_mlp"] if "raw_mlp" in large_fm_dfs[variant].index else large_fm_dfs[variant].iloc[0]
    mean = row[metric_mean]
    std = row[metric_std] if metric_std in row.index else 0.0
    ax.bar(large_fm_x[variant], mean, bar_w * 1.2, yerr=std, capsize=3,
           color="#8172B3", alpha=0.88,
           label="_nolegend_",
           error_kw={"linewidth": 1.2})

for x_pos, (_name, mean, sd) in zip(ref_x, ref_models):
    ax.bar(x_pos, mean, bar_w * 1.2, yerr=sd, capsize=3,
           color="#999999", alpha=0.88, label="_nolegend_",
           error_kw={"linewidth": 1.2} if sd is not None else {})

sc_lo, sc_hi = centers["pretrain_sc"] - 0.45, centers["preadapt_sc"] + 0.45
bulk_lo, bulk_hi = centers["pretrain_bulk"] - 0.45, centers["preadapt_bulk"] + 0.45
baseline_lo = (min(baseline_x.values()) - 0.42) if baseline_x else None
baseline_hi = (max(baseline_x.values()) + 0.42) if baseline_x else None
large_fm_lo = (min(large_fm_x.values()) - 0.42) if large_fm_x else None
large_fm_hi = (max(large_fm_x.values()) + 0.42) if large_fm_x else None
ref_lo, ref_hi = ref_x[0] - 0.42, ref_x[-1] + 0.42
ax.axvspan(sc_lo, sc_hi, alpha=0.07, color="steelblue", zorder=0)
ax.axvspan(bulk_lo, bulk_hi, alpha=0.07, color="salmon", zorder=0)
if baseline_x:
    ax.axvspan(baseline_lo, baseline_hi, alpha=0.06, color="darkgray", zorder=0)
if large_fm_x:
    ax.axvspan(large_fm_lo, large_fm_hi, alpha=0.07, color="#8172B3", zorder=0)
ax.axvspan(ref_lo, ref_hi, alpha=0.05, color="gray", zorder=0)

y_lim = (0, 1.16)
ax.set_ylim(*y_lim)
ax.text((sc_lo + sc_hi) / 2, y_lim[1] - 0.015, "sc pre-trained", ha="center", va="top", fontsize=9, color="steelblue", style="italic")
ax.text((bulk_lo + bulk_hi) / 2, y_lim[1] - 0.015, "bulk pre-trained", ha="center", va="top", fontsize=9, color="firebrick", style="italic")
if baseline_x:
    ax.text((baseline_lo + baseline_hi) / 2, y_lim[1] - 0.015, "baselines", ha="center", va="top", fontsize=9, color="dimgray", style="italic")
if large_fm_x:
    ax.text((large_fm_lo + large_fm_hi) / 2, y_lim[1] - 0.015, "large FMs", ha="center", va="top", fontsize=9, color="#8172B3", style="italic")
ax.text((ref_lo + ref_hi) / 2, y_lim[1] - 0.015, "reference models", ha="center", va="top", fontsize=9, color="dimgray", style="italic")

large_fm_label_map = {variant: label for variant, label in large_fm_specs}
tick_pos = (
    list(centers.values())
    + list(baseline_x.values())
    + [large_fm_x[v] for v in available_large_fms]
    + ref_x
)
tick_labels = (
    ["Random\ninit", "Pretrain\nsc", "Preadapt\nsc", "Pretrain\nbulk", "Preadapt\nbulk"]
    + [dict(baseline_specs)[v] for v in available_baselines]
    + [large_fm_label_map[v] for v in available_large_fms]
    + [name for name, _mean, _sd in ref_models]
)
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, fontsize=8.8)
ax.set_ylabel("Weighted F1", fontsize=11)
ax.set_title("Cancer Classification (33 types) — Weighted F1", fontsize=12)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9.5, framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout(rect=(0, 0, 0.86, 1))
plt.show()


### Bulk pretraining dataset-size downstream comparison

In [ ]:
task = "canc_type_class_33"
base = str(OUTPUT_DIR / task)
metric_mean = "f1_weighted_mean"
metric_std = "f1_weighted_std"

bulk_downstream_specs = [
    ("10k", "head_only_pretrain_bulk_10k"),
    ("50k", "head_only_pretrain_bulk_50k"),
    ("100k", "head_only_pretrain_bulk_100k"),
    ("200k", "head_only_pretrain_bulk_200k"),
    ("400k", "head_only_pretrain_bulk_400k"),
    ("Full", None),
]


def load_bulk_downstream_result(label, variant):
    if label == "Full":
        metadata_path = f"{base}/head_only/{task}_head_only_run_metadata.json"
        candidates = [
            f"{base}/head_only/{task}_head_only_pretrain_bulk_evaluation_metrics.csv",
            f"{base}/head_only/{task}_head_only_evaluation_metrics.csv",
        ]
    else:
        prefix = f"{task}_{variant}"
        metadata_path = f"{base}/{variant}/{prefix}_run_metadata.json"
        candidates = [
            f"{base}/{variant}/{prefix}_evaluation_metrics.csv",
            f"{base}/{variant}/{prefix}_pretrain_bulk_evaluation_metrics.csv",
        ]

    if not os.path.exists(metadata_path):
        return None, None, None
    with open(metadata_path) as handle:
        metadata = __import__("json").load(handle)
    if (
        metadata.get("head_only_backbone_eval") is not True
        or metadata.get("run_status") != "complete"
    ):
        print(f"Skipping incomplete or incompatible head-only result: {label}")
        return None, None, None
    fold_fingerprint = metadata.get("cv_fold_fingerprint")
    if not fold_fingerprint:
        print(f"Skipping head-only result without fold fingerprint: {label}")
        return None, None, None

    for path in candidates:
        if not os.path.exists(path):
            continue
        frame = pd.read_csv(path, comment="#")
        if "model" in frame.columns:
            frame = frame[frame["model"] == "pretrain_bulk"]
        if not frame.empty and metric_mean in frame.columns:
            return frame.iloc[0], path, fold_fingerprint
    return None, None, None


bulk_downstream_rows = []
for label, variant in bulk_downstream_specs:
    row, path, fold_fingerprint = load_bulk_downstream_result(label, variant)
    if row is not None:
        bulk_downstream_rows.append({
            "label": label,
            "mean": float(row[metric_mean]),
            "std": float(row[metric_std]) if metric_std in row.index else 0.0,
            "path": path,
            "fold_fingerprint": fold_fingerprint,
        })

if not bulk_downstream_rows:
    raise FileNotFoundError(f"No complete bulk-size head-only results found under {base}")
if len({row["fold_fingerprint"] for row in bulk_downstream_rows}) != 1:
    raise ValueError("Bulk-size head-only results do not use identical CV folds")

x = np.arange(len(bulk_downstream_rows))
means = np.asarray([row["mean"] for row in bulk_downstream_rows])
stds = np.asarray([row["std"] for row in bulk_downstream_rows])
bar_colors = plt.get_cmap("viridis")(
    np.linspace(0.12, 0.88, len(bulk_downstream_rows))
)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(
    x, means, yerr=stds, capsize=4, width=0.68,
    color=bar_colors, alpha=0.9, error_kw={"linewidth": 1.2},
)
for bar, mean in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2, mean + 0.018, f"{mean:.3f}",
        ha="center", va="bottom", fontsize=9,
    )

ax.set_xticks(x)
ax.set_xticklabels([row["label"] for row in bulk_downstream_rows])
ax.set_ylim(0, 1.08)
ax.set_xlabel("Bulk pretraining samples", fontsize=11)
ax.set_ylabel("Weighted F1", fontsize=11)
ax.set_title("Cancer classification (33 types): head-only performance by pretraining size", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


### Validation-loss convergence


In [ ]:
plot_finetuning_convergence(
    "canc_type_class_33",
    "Cancer classification (33 types)",
    "Validation weighted cross-entropy loss",
)


## Deconvolution

In [ ]:
task = "deconv"
base = str(OUTPUT_DIR / task)
modes = ["head_only", "adapters", "full_ft", "pca_rf"]
mode_labels = {
    "head_only": "Head", "adapters": "Adapters",
    "full_ft": "Full", "pca_rf": "PCA+RF",
}
mode_colors = {
    "head_only": "#4C72B0", "adapters": "#DD8452",
    "full_ft": "#55A868", "pca_rf": "#8172B3",
}
model_keys = ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]
metric_mean = "mae_mean"
metric_std = "mae_std"
baseline_specs = [
    ("raw_mlp_all_genes", "RE MLP\nAll"),
    ("raw_mlp_hvg1199", "RE MLP\nMAD"),
    ("raw_pca_rf_all_genes", "RE PCA+RF\nAll"),
    ("raw_pca_rf_hvg1199", "RE PCA+RF\nMAD"),
]
large_fm_specs = [
    ("scgpt_pca_rf", "scGPT"),
    ("scgpt_preadapt_pca_rf", "Preadapted\nscGPT"),
    ("bulkformer_pca_rf", "BulkFormer"),
]


def load_deconv_eval(path):
    if os.path.exists(path):
        return pd.read_csv(path, comment="#").set_index("model")
    return None


def deconv_output_is_current(variant):
    metadata_path = f"{base}/{variant}/{task}_{variant}_run_metadata.json"
    if not os.path.exists(metadata_path):
        return False
    with open(metadata_path) as handle:
        metadata = json.load(handle)
    mse_trained_variant = variant in {
        "head_only", "adapters", "full_ft",
        "raw_mlp_all_genes", "raw_mlp_hvg1199",
    }
    return (
        metadata.get("run_status") == "complete"
        and metadata.get("cv_fold_fingerprint")
        and metadata.get("gene_selection") == "mad"
        and metadata.get("expression_transform") == "log1p"
        and (
            not mse_trained_variant
            or (
                metadata.get("training_objective") == "mse"
                and metadata.get("primary_evaluation_metric") == "mae"
            )
        )
    )


def load_deconv_mode(mode):
    if not deconv_output_is_current(mode):
        return None
    combined = f"{base}/{mode}/{task}_{mode}_evaluation_metrics.csv"
    frame = load_deconv_eval(combined)
    if frame is not None:
        return frame
    parts = [
        pd.read_csv(
            f"{base}/{mode}/{task}_{mode}_{model}_evaluation_metrics.csv",
            comment="#",
        )
        for model in model_keys
        if os.path.exists(f"{base}/{mode}/{task}_{mode}_{model}_evaluation_metrics.csv")
    ]
    return pd.concat(parts).set_index("model") if parts else None


def load_deconv_variants(specs, require_complete=False):
    loaded = {}
    for variant, _label in specs:
        if not deconv_output_is_current(variant):
            print(f"Skipping stale or partial external baseline: {variant}")
            continue
        path = f"{base}/{variant}/{task}_{variant}_evaluation_metrics.csv"
        frame = load_deconv_eval(path)
        if frame is not None and metric_mean in frame.columns:
            loaded[variant] = frame
    return loaded


mode_frames = {mode: load_deconv_mode(mode) for mode in modes}
available_modes = [mode for mode in modes if mode_frames[mode] is not None]
baseline_frames = load_deconv_variants(baseline_specs)
large_fm_frames = load_deconv_variants(large_fm_specs, require_complete=True)
available_baselines = [variant for variant, _ in baseline_specs if variant in baseline_frames]
available_large_fms = [variant for variant, _ in large_fm_specs if variant in large_fm_frames]
if not available_modes and not available_baselines and not available_large_fms:
    raise FileNotFoundError(f"No evaluation metric files found under {base}")

bar_width = 0.18
centers = {
    "random_init": 0.0, "pretrain_sc": 1.1, "preadapt_sc": 1.9,
    "pretrain_bulk": 3.0, "preadapt_bulk": 3.8,
}
baseline_x = {
    variant: 5.0 + index * 0.78
    for index, variant in enumerate(available_baselines)
}
large_start = max(baseline_x.values(), default=4.05) + 0.95
large_x = {
    variant: large_start + index * 0.78
    for index, variant in enumerate(available_large_fms)
}

mean_baseline_value = None
mean_baseline_std = 0.0
for frame in [*mode_frames.values(), *baseline_frames.values(), *large_fm_frames.values()]:
    if frame is None or "mean_baseline_mae_mean" not in frame.columns:
        continue
    mean_baseline_value = float(frame.iloc[0]["mean_baseline_mae_mean"])
    if "mean_baseline_mae_std" in frame.columns:
        mean_baseline_std = float(frame.iloc[0]["mean_baseline_mae_std"])
    break
mean_baseline_x = max([*large_x.values(), *baseline_x.values(), 3.8]) + 0.95

fig, ax = plt.subplots(figsize=(17, 5.5))
added_modes = set()
plotted_values = []
for mode_index, mode in enumerate(available_modes):
    offset = (mode_index - (len(available_modes) - 1) / 2) * bar_width
    frame = mode_frames[mode]
    for model, center in centers.items():
        if model not in frame.index or metric_mean not in frame.columns:
            continue
        mean = float(frame.loc[model, metric_mean])
        std = float(frame.loc[model, metric_std]) if metric_std in frame.columns else 0.0
        ax.bar(
            center + offset, mean, bar_width, yerr=std, capsize=3,
            color=mode_colors[mode], alpha=0.88,
            label=mode_labels[mode] if mode not in added_modes else "",
            error_kw={"linewidth": 1.2},
        )
        plotted_values.append(mean + std)
        added_modes.add(mode)

for variant, _label in baseline_specs:
    if variant not in baseline_frames:
        continue
    row = baseline_frames[variant].iloc[0]
    mean = float(row[metric_mean])
    std = float(row[metric_std]) if metric_std in row.index else 0.0
    ax.bar(
        baseline_x[variant], mean, bar_width * 1.2, yerr=std, capsize=3,
        color="#8C8C8C", alpha=0.88,
        label="Simple baselines" if variant == available_baselines[0] else "",
        error_kw={"linewidth": 1.2},
    )
    plotted_values.append(mean + std)

for variant, _label in large_fm_specs:
    if variant not in large_fm_frames:
        continue
    row = large_fm_frames[variant].iloc[0]
    mean = float(row[metric_mean])
    std = float(row[metric_std]) if metric_std in row.index else 0.0
    ax.bar(
        large_x[variant], mean, bar_width * 1.2, yerr=std, capsize=3,
        color="#8172B3", alpha=0.88, label="_nolegend_",
        error_kw={"linewidth": 1.2},
    )
    plotted_values.append(mean + std)

if mean_baseline_value is not None:
    ax.bar(
        mean_baseline_x, mean_baseline_value, bar_width * 1.2,
        yerr=mean_baseline_std, capsize=3, color="#B0B0B0", alpha=0.88,
        label="_nolegend_", error_kw={"linewidth": 1.2},
    )
    plotted_values.append(mean_baseline_value + mean_baseline_std)

sc_span = (centers["pretrain_sc"] - 0.45, centers["preadapt_sc"] + 0.45)
bulk_span = (centers["pretrain_bulk"] - 0.45, centers["preadapt_bulk"] + 0.45)
ax.axvspan(*sc_span, alpha=0.07, color="steelblue", zorder=0)
ax.axvspan(*bulk_span, alpha=0.07, color="salmon", zorder=0)
if baseline_x:
    ax.axvspan(min(baseline_x.values()) - 0.42, max(baseline_x.values()) + 0.42,
               alpha=0.06, color="darkgray", zorder=0)
if large_x:
    ax.axvspan(min(large_x.values()) - 0.42, max(large_x.values()) + 0.42,
               alpha=0.07, color="#8172B3", zorder=0)

y_max = max(plotted_values, default=0.8) * 1.22
ax.set_ylim(0, y_max)
label_y = ax.get_ylim()[1] * 0.98
ax.text(np.mean(sc_span), label_y, "sc pre-trained", ha="center", va="top",
        fontsize=9, color="steelblue", style="italic")
ax.text(np.mean(bulk_span), label_y, "bulk pre-trained", ha="center", va="top",
        fontsize=9, color="firebrick", style="italic")

label_map = dict([*baseline_specs, *large_fm_specs])
tick_positions = (
    list(centers.values()) + list(baseline_x.values())
    + [large_x[value] for value in available_large_fms]
)
tick_labels = (
    ["Random\ninit", "Pretrain\nsc", "Preadapt\nsc", "Pretrain\nbulk", "Preadapt\nbulk"]
    + [label_map[value] for value in available_baselines]
    + [label_map[value] for value in available_large_fms]
)
if mean_baseline_value is not None:
    tick_positions.append(mean_baseline_x)
    tick_labels.append("Training-fold\nmean")
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, fontsize=8.8)
ax.set_ylabel("Mean absolute error (lower is better)", fontsize=11)
ax.set_title("Cell-type deconvolution", fontsize=12)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9, framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout(rect=(0, 0, 0.86, 1))
plt.show()



In [ ]:
# Load directly from disk so stale notebook functions or cached frames cannot affect the plots.
task = "deconv"
base = str(OUTPUT_DIR / task)
modes = ["head_only", "adapters", "full_ft", "pca_rf"]
mode_labels = {
    "head_only": "Head", "adapters": "Adapters",
    "full_ft": "Full", "pca_rf": "PCA+RF",
}
mode_colors = {
    "head_only": "#4C72B0", "adapters": "#DD8452",
    "full_ft": "#55A868", "pca_rf": "#8172B3",
}
baseline_specs = [
    ("raw_mlp_all_genes", "RE MLP\nAll"),
    ("raw_mlp_hvg1199", "RE MLP\nMAD"),
    ("raw_pca_rf_all_genes", "RE PCA+RF\nAll"),
    ("raw_pca_rf_hvg1199", "RE PCA+RF\nMAD"),
]
large_fm_specs = [
    ("scgpt_pca_rf", "scGPT"),
    ("scgpt_preadapt_pca_rf", "Preadapted\nscGPT"),
    ("bulkformer_pca_rf", "BulkFormer"),
]
bar_width = 0.18
centers = {
    "random_init": 0.0, "pretrain_sc": 1.1, "preadapt_sc": 1.9,
    "pretrain_bulk": 3.0, "preadapt_bulk": 3.8,
}
mse_variants = {
    "head_only", "adapters", "full_ft",
    "raw_mlp_all_genes", "raw_mlp_hvg1199",
}
required_metric_columns = {
    "model", "rmse_mean", "rmse_std", "mean_baseline_rmse_mean",
    "mean_baseline_rmse_std",
    "mean_cell_type_pearson_across_samples_mean",
    "mean_cell_type_pearson_across_samples_std",
}


def load_deconv_plot_frame(variant):
    metadata_path = f"{base}/{variant}/{task}_{variant}_run_metadata.json"
    metrics_path = f"{base}/{variant}/{task}_{variant}_evaluation_metrics.csv"
    if not os.path.exists(metadata_path) or not os.path.exists(metrics_path):
        raise FileNotFoundError(f"Missing completed deconvolution output for {variant}")
    with open(metadata_path) as handle:
        metadata = json.load(handle)
    expected_objective = "mse" if variant in mse_variants else "random_forest_squared_error"
    expected_metadata = {
        "run_status": "complete",
        "gene_selection": "mad",
        "expression_transform": "log1p",
        "training_objective": expected_objective,
        "primary_evaluation_metric": "mae",
    }
    mismatches = [
        f"{key}={metadata.get(key)!r} (expected {expected!r})"
        for key, expected in expected_metadata.items()
        if metadata.get(key) != expected
    ]
    if not metadata.get("cv_fold_fingerprint"):
        mismatches.append("missing cv_fold_fingerprint")
    if mismatches:
        raise ValueError(f"Incompatible deconvolution output for {variant}: {', '.join(mismatches)}")
    frame = pd.read_csv(metrics_path, comment="#")
    missing_columns = sorted(required_metric_columns - set(frame.columns))
    if missing_columns:
        raise ValueError(f"{metrics_path} lacks columns: {missing_columns}")
    return frame.set_index("model"), metadata["cv_fold_fingerprint"]


plot_variants = list(dict.fromkeys([
    *modes,
    *[variant for variant, _ in baseline_specs],
    *[variant for variant, _ in large_fm_specs],
]))
loaded_plot_results = {variant: load_deconv_plot_frame(variant) for variant in plot_variants}
fold_fingerprints = {result[1] for result in loaded_plot_results.values()}
if len(fold_fingerprints) != 1:
    raise ValueError("Deconvolution results do not use identical CV folds")
mode_frames = {mode: loaded_plot_results[mode][0] for mode in modes}
baseline_frames = {variant: loaded_plot_results[variant][0] for variant, _ in baseline_specs}
large_fm_frames = {variant: loaded_plot_results[variant][0] for variant, _ in large_fm_specs}
available_modes = list(modes)
available_baselines = [variant for variant, _ in baseline_specs]
available_large_fms = [variant for variant, _ in large_fm_specs]
baseline_x = {
    variant: 5.0 + index * 0.78
    for index, variant in enumerate(available_baselines)
}
large_start = max(baseline_x.values(), default=4.05) + 0.95
large_x = {
    variant: large_start + index * 0.78
    for index, variant in enumerate(available_large_fms)
}
mean_baseline_x = max([*large_x.values(), *baseline_x.values(), 3.8]) + 0.95
label_map = dict([*baseline_specs, *large_fm_specs])
sc_span = (centers["pretrain_sc"] - 0.45, centers["preadapt_sc"] + 0.45)
bulk_span = (centers["pretrain_bulk"] - 0.45, centers["preadapt_bulk"] + 0.45)

additional_deconv_metrics = [
    {
        "mean": "rmse_mean",
        "std": "rmse_std",
        "baseline_mean": "mean_baseline_rmse_mean",
        "baseline_std": "mean_baseline_rmse_std",
        "ylabel": "Root mean squared error (lower is better)",
        "title": "Cell-type deconvolution: RMSE",
        "higher_is_better": False,
    },
    {
        "mean": "total_variation_distance_mean",
        "std": "total_variation_distance_std",
        "baseline_mean": "mean_baseline_total_variation_distance_mean",
        "baseline_std": "mean_baseline_total_variation_distance_std",
        "ylabel": "Total variation distance (lower is better)",
        "title": "Cell-type deconvolution: total variation distance",
        "higher_is_better": False,
    },
    {
        "mean": "mean_cell_type_pearson_across_samples_mean",
        "std": "mean_cell_type_pearson_across_samples_std",
        "baseline_mean": None,
        "baseline_std": None,
        "ylabel": "Mean cell-type PCC across samples (higher is better)",
        "title": "Cell-type deconvolution: cell-type PCC",
        "higher_is_better": True,
    },
]


def plot_deconv_summary_metric(spec):
    fig, ax = plt.subplots(figsize=(17, 5.5))
    added_modes = set()
    plotted_highs = []
    plotted_lows = []
    for mode_index, mode in enumerate(available_modes):
        offset = (mode_index - (len(available_modes) - 1) / 2) * bar_width
        frame = mode_frames.get(mode)
        if frame is None or frame.empty:
            continue
        for model, center in centers.items():
            if model not in frame.index or spec["mean"] not in frame.columns:
                continue
            mean = float(frame.loc[model, spec["mean"]])
            std = float(frame.loc[model, spec["std"]]) if spec["std"] in frame.columns else 0.0
            ax.bar(
                center + offset, mean, bar_width, yerr=std, capsize=3,
                color=mode_colors[mode], alpha=0.88,
                label=mode_labels[mode] if mode not in added_modes else "",
                error_kw={"linewidth": 1.2},
            )
            plotted_highs.append(mean + std)
            plotted_lows.append(mean - std)
            added_modes.add(mode)

    simple_baseline_added = False
    for variant, _label in baseline_specs:
        frame = baseline_frames.get(variant)
        if frame is None or frame.empty:
            continue
        row = frame.iloc[0]
        if spec["mean"] not in row.index:
            continue
        mean = float(row[spec["mean"]])
        std = float(row[spec["std"]]) if spec["std"] in row.index else 0.0
        ax.bar(
            baseline_x[variant], mean, bar_width * 1.2, yerr=std, capsize=3,
            color="#8C8C8C", alpha=0.88,
            label="Simple baselines" if not simple_baseline_added else "",
            error_kw={"linewidth": 1.2},
        )
        plotted_highs.append(mean + std)
        plotted_lows.append(mean - std)
        simple_baseline_added = True

    for variant, _label in large_fm_specs:
        frame = large_fm_frames.get(variant)
        if frame is None or frame.empty:
            continue
        row = frame.iloc[0]
        if spec["mean"] not in row.index:
            continue
        mean = float(row[spec["mean"]])
        std = float(row[spec["std"]]) if spec["std"] in row.index else 0.0
        ax.bar(
            large_x[variant], mean, bar_width * 1.2, yerr=std, capsize=3,
            color="#8172B3", alpha=0.88, label="_nolegend_",
            error_kw={"linewidth": 1.2},
        )
        plotted_highs.append(mean + std)
        plotted_lows.append(mean - std)

    baseline_value = None
    baseline_std = 0.0
    if spec["baseline_mean"] is not None:
        for frame in [*mode_frames.values(), *baseline_frames.values(), *large_fm_frames.values()]:
            if frame is None or spec["baseline_mean"] not in frame.columns:
                continue
            baseline_value = float(frame.iloc[0][spec["baseline_mean"]])
            if spec["baseline_std"] in frame.columns:
                baseline_std = float(frame.iloc[0][spec["baseline_std"]])
            break
    if baseline_value is not None:
        ax.bar(
            mean_baseline_x, baseline_value, bar_width * 1.2,
            yerr=baseline_std, capsize=3, color="#B0B0B0", alpha=0.88,
            label="_nolegend_", error_kw={"linewidth": 1.2},
        )
        plotted_highs.append(baseline_value + baseline_std)
        plotted_lows.append(baseline_value - baseline_std)

    ax.axvspan(*sc_span, alpha=0.07, color="steelblue", zorder=0)
    ax.axvspan(*bulk_span, alpha=0.07, color="salmon", zorder=0)
    if baseline_x:
        ax.axvspan(min(baseline_x.values()) - 0.42, max(baseline_x.values()) + 0.42,
                   alpha=0.06, color="darkgray", zorder=0)
    if large_x:
        ax.axvspan(min(large_x.values()) - 0.42, max(large_x.values()) + 0.42,
                   alpha=0.07, color="#8172B3", zorder=0)

    if spec["higher_is_better"]:
        y_min = min(0.0, min(plotted_lows, default=0.0) * 1.1)
        y_max = min(1.0, max(plotted_highs, default=1.0) * 1.18)
        ax.axhline(0.0, color="#777777", linewidth=0.8, zorder=0)
    else:
        y_min = 0.0
        y_max = max(plotted_highs, default=1.0) * 1.22
    ax.set_ylim(y_min, y_max)
    label_y = y_max - (y_max - y_min) * 0.02
    ax.text(np.mean(sc_span), label_y, "sc pre-trained", ha="center", va="top",
            fontsize=9, color="steelblue", style="italic")
    ax.text(np.mean(bulk_span), label_y, "bulk pre-trained", ha="center", va="top",
            fontsize=9, color="firebrick", style="italic")

    tick_positions = (
        list(centers.values()) + list(baseline_x.values())
        + [large_x[value] for value in available_large_fms]
    )
    tick_labels = (
        ["Random\ninit", "Pretrain\nsc", "Preadapt\nsc", "Pretrain\nbulk", "Preadapt\nbulk"]
        + [label_map[value] for value in available_baselines]
        + [label_map[value] for value in available_large_fms]
    )
    if baseline_value is not None:
        tick_positions.append(mean_baseline_x)
        tick_labels.append("Training-fold\nmean")
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=8.8)
    ax.set_ylabel(spec["ylabel"], fontsize=11)
    ax.set_title(spec["title"], fontsize=12)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9, framealpha=0.9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout(rect=(0, 0, 0.86, 1))
    plt.show()


for deconv_metric in additional_deconv_metrics:
    plot_deconv_summary_metric(deconv_metric)


### Bulk pretraining dataset-size downstream comparison


In [ ]:
task = "deconv"
base = str(OUTPUT_DIR / task)
metric_mean = "mae_mean"
metric_std = "mae_std"
bulk_specs = [
    ("10k", "head_only_pretrain_bulk_10k"),
    ("50k", "head_only_pretrain_bulk_50k"),
    ("100k", "head_only_pretrain_bulk_100k"),
    ("200k", "head_only_pretrain_bulk_200k"),
    ("400k", "head_only_pretrain_bulk_400k"),
    ("Full", None),
]


def load_deconv_bulk_size(label, variant):
    if label == "Full":
        output_dir = "head_only"
        prefix = f"{task}_head_only"
    else:
        output_dir = variant
        prefix = f"{task}_{variant}"
    metadata_path = f"{base}/{output_dir}/{prefix}_run_metadata.json"
    metrics_path = f"{base}/{output_dir}/{prefix}_evaluation_metrics.csv"
    if not os.path.exists(metadata_path) or not os.path.exists(metrics_path):
        return None
    with open(metadata_path) as handle:
        metadata = json.load(handle)
    if (
        metadata.get("head_only_backbone_eval") is not True
        or metadata.get("run_status") != "complete"
        or not metadata.get("cv_fold_fingerprint")
        or metadata.get("gene_selection") != "mad"
        or metadata.get("expression_transform") != "log1p"
        or metadata.get("training_objective") != "mse"
        or metadata.get("primary_evaluation_metric") != "mae"
    ):
        print(f"Skipping incomplete or incompatible result: {label}")
        return None
    frame = pd.read_csv(metrics_path, comment="#")
    frame = frame[frame["model"] == "pretrain_bulk"]
    if frame.empty or metric_mean not in frame.columns:
        return None
    row = frame.iloc[0]
    return {
        "label": label,
        "mean": float(row[metric_mean]),
        "std": float(row[metric_std]) if metric_std in row.index else 0.0,
        "rmse_mean": float(row["rmse_mean"]),
        "rmse_std": float(row["rmse_std"]) if "rmse_std" in row.index else 0.0,
        "pcc_mean": float(row["mean_cell_type_pearson_across_samples_mean"]),
        "pcc_std": float(row["mean_cell_type_pearson_across_samples_std"])
        if "mean_cell_type_pearson_across_samples_std" in row.index else 0.0,
        "fold_fingerprint": metadata["cv_fold_fingerprint"],
    }


bulk_rows = [row for label, variant in bulk_specs
             if (row := load_deconv_bulk_size(label, variant)) is not None]
if not bulk_rows:
    raise FileNotFoundError(f"No complete bulk-size head-only results found under {base}")
if len({row["fold_fingerprint"] for row in bulk_rows}) != 1:
    raise ValueError("Bulk-size deconvolution results do not use identical CV folds")

x = np.arange(len(bulk_rows))
means = np.asarray([row["mean"] for row in bulk_rows])
stds = np.asarray([row["std"] for row in bulk_rows])
bar_colors = plt.get_cmap("viridis")(np.linspace(0.12, 0.88, len(bulk_rows)))
fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(
    x, means, yerr=stds, capsize=4, width=0.68,
    color=bar_colors, alpha=0.9, error_kw={"linewidth": 1.2},
)
offset = float(np.max(means + stds)) * 0.025
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, mean + offset, f"{mean:.5f}",
            ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([row["label"] for row in bulk_rows])
ax.set_ylim(0, float(np.max(means + stds)) * 1.2)
ax.set_xlabel("Bulk pretraining samples", fontsize=11)
ax.set_ylabel("Mean absolute error (lower is better)", fontsize=11)
ax.set_title("Deconvolution: head-only performance by pretraining size", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()



In [ ]:
bulk_size_additional_metrics = [
    ("rmse_mean", "rmse_std", "Root mean squared error (lower is better)",
     "Deconvolution RMSE by pretraining size", ".5f", False),
    ("pcc_mean", "pcc_std", "Mean cell-type PCC across samples (higher is better)",
     "Deconvolution cell-type PCC by pretraining size", ".3f", True),
]

for mean_key, std_key, ylabel, title, value_format, higher_is_better in bulk_size_additional_metrics:
    means = np.asarray([row[mean_key] for row in bulk_rows])
    stds = np.asarray([row[std_key] for row in bulk_rows])
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    bars = ax.bar(
        x, means, yerr=stds, capsize=4, width=0.68,
        color=bar_colors, alpha=0.9, error_kw={"linewidth": 1.2},
    )
    value_range = float(np.max(means + stds) - min(0.0, np.min(means - stds)))
    offset = max(value_range * 0.025, np.finfo(float).eps)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, mean + offset, format(mean, value_format),
                ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([row["label"] for row in bulk_rows])
    if higher_is_better:
        y_min = min(0.0, float(np.min(means - stds)) * 1.1)
        y_max = min(1.0, float(np.max(means + stds)) * 1.2)
        ax.axhline(0.0, color="#777777", linewidth=0.8, zorder=0)
    else:
        y_min = 0.0
        y_max = float(np.max(means + stds)) * 1.2
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("Bulk pretraining samples", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()


### Validation-loss convergence


In [ ]:
task = "deconv"
base = str(OUTPUT_DIR / task)
curve_specs = [
    ("head_only", "Head", ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]),
    ("adapters", "Adapters", ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]),
    ("full_ft", "Full", ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]),
    ("raw_mlp_all_genes", "RE MLP, All", ["raw_mlp"]),
    ("raw_mlp_hvg1199", "RE MLP, MAD", ["raw_mlp"]),
]
model_colors = {
    "random_init": "#4C72B0", "pretrain_sc": "#55A868",
    "preadapt_sc": "#2C7FB8", "pretrain_bulk": "#DD8452",
    "preadapt_bulk": "#C44E52", "raw_mlp": "#666666",
}
available_curve_specs = []
for variant, title, models in curve_specs:
    series = {}
    for model in models:
        path = f"{base}/{variant}/{task}_{variant}_{model}_training_curves.csv"
        if os.path.exists(path):
            frame = pd.read_csv(path)
            if {"epoch", "fold", "validation_loss"}.issubset(frame.columns):
                series[model] = frame
    if series:
        available_curve_specs.append((variant, title, series))
if not available_curve_specs:
    raise FileNotFoundError(f"No deconvolution training curves found under {base}")

n_columns = 2
n_rows = int(np.ceil(len(available_curve_specs) / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(12, 4.0 * n_rows), squeeze=False)
for ax, (_variant, title, series) in zip(axes.flat, available_curve_specs):
    for model, frame in series.items():
        summary = frame.groupby("epoch")["validation_loss"].agg(["mean", "std"]).reset_index()
        std = summary["std"].fillna(0.0).to_numpy()
        epochs = summary["epoch"].to_numpy()
        means = summary["mean"].to_numpy()
        ax.plot(epochs, means, marker="o", markersize=3, lw=1.8,
                color=model_colors[model], label=model.replace("_", " "))
        ax.fill_between(epochs, means - std, means + std,
                        color=model_colors[model], alpha=0.12)
    ax.set_yscale("log")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation MSE loss")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8, framealpha=0.9)
for ax in axes.flat[len(available_curve_specs):]:
    ax.set_visible(False)
fig.suptitle("Deconvolution convergence across folds", fontsize=13)
plt.tight_layout()
plt.show()



## Disease classification

In [ ]:
bulkformer_weighted_f1_mean = 0.939
geneformer_weighted_f1_mean = 0.749
genecompass_weighted_f1_mean = 0.882
scgpt_weighted_f1_mean = 0.885
scfoundation_weighted_f1_mean = 0.874
sclong_weighted_f1_mean = 0.81


In [ ]:
task = "disease_class"
base = str(OUTPUT_DIR / task)
modes = ["head_only", "adapters", "full_ft", "pca_rf"]
mode_labels = {"head_only": "Head", "adapters": "Adapters", "full_ft": "Full", "pca_rf": "PCA+RF"}
colors  = {"head_only": "#4C72B0", "adapters": "#DD8452", "full_ft": "#55A868", "pca_rf": "#8172B3"}
model_keys = ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]
metric_mean = "f1_weighted_mean"
metric_std = "f1_weighted_std"

baseline_specs = [
    ("raw_mlp_all_genes", "RE MLP\nAll"),
    ("raw_mlp_hvg1199", "RE MLP\nMAD"),
    ("raw_pca_rf_all_genes", "RE PCA+RF\nAll"),
    ("raw_pca_rf_hvg1199", "RE PCA+RF\nMAD"),
]
large_fm_specs = [
    ("scgpt_pca_rf", "scGPT"),
    ("scgpt_preadapt_pca_rf", "Preadapted\nscGPT"),
    ("bulkformer_pca_rf", "BulkFormer"),
]

def load_eval_csv(path):
    if os.path.exists(path):
        return pd.read_csv(path, comment="#").set_index("model")
    return None


def load_mode(mode):
    if not head_only_output_is_current(base, task, mode):
        return None
    combined = f"{base}/{mode}/{task}_{mode}_evaluation_metrics.csv"
    df = load_eval_csv(combined)
    if df is not None:
        return df
    parts = [
        pd.read_csv(f"{base}/{mode}/{task}_{mode}_{m}_evaluation_metrics.csv", comment="#")
        for m in model_keys
        if os.path.exists(f"{base}/{mode}/{task}_{mode}_{m}_evaluation_metrics.csv")
    ]
    return pd.concat(parts).set_index("model") if parts else None


def load_variant_metrics(specs, metric_col):
    loaded = {}
    for spec in specs:
        canonical = spec[0]
        if canonical in {'scgpt_pca_rf', 'scgpt_preadapt_pca_rf', 'bulkformer_pca_rf'} and not completed_output_is_current(base, task, canonical):
            print(f'Skipping stale or partial external baseline: {canonical}')
            continue
        path = f"{base}/{canonical}/{task}_{canonical}_evaluation_metrics.csv"
        df = load_eval_csv(path)
        if df is not None and metric_col in df.columns:
            loaded[canonical] = df
    return loaded


dfs = {mode: load_mode(mode) for mode in modes}
available_modes = [mode for mode in modes if dfs[mode] is not None]
baseline_dfs = load_variant_metrics(baseline_specs, metric_mean)
large_fm_dfs = load_variant_metrics(large_fm_specs, metric_mean)
available_baselines = [variant for variant, _ in baseline_specs if variant in baseline_dfs]
available_large_fms = [variant for variant, _label in large_fm_specs if variant in large_fm_dfs]
if not available_modes and not available_baselines and not available_large_fms:
    raise FileNotFoundError(f"No evaluation metric files found under {base}")

bar_w = 0.18
centers = {
    "random_init":  0.0,
    "pretrain_sc":  1.1,
    "preadapt_sc":  1.9,
    "pretrain_bulk": 3.0,
    "preadapt_bulk": 3.8,
}
baseline_start = 5.0
baseline_spacing = 0.78
baseline_x = {variant: baseline_start + i * baseline_spacing for i, variant in enumerate(available_baselines)}
large_fm_start = (max(baseline_x.values()) + 0.95) if baseline_x else 5.0
large_fm_spacing = 0.78
large_fm_x = {variant: large_fm_start + i * large_fm_spacing for i, variant in enumerate(available_large_fms)}
rightmost_nonref = max(
    list(baseline_x.values())
    + list(large_fm_x.values())
    + [centers["preadapt_bulk"]]
)
ref_models = [
    ("BulkFormer\n(reported)", bulkformer_weighted_f1_mean, None),
    ("GeneFormer", geneformer_weighted_f1_mean, None),
    ("GeneCompass", genecompass_weighted_f1_mean, None),
    ("scGPT\n(reported)", scgpt_weighted_f1_mean, None),
    ("scFoundation", scfoundation_weighted_f1_mean, None),
    ("scLong", sclong_weighted_f1_mean, None),
]
ref_start = rightmost_nonref + 1.05
ref_spacing = 0.78
ref_x = [ref_start + i * ref_spacing for i in range(len(ref_models))]

fig, ax = plt.subplots(figsize=(20, 5.5))

added = set()
for m_idx, mode in enumerate(available_modes):
    offset = (m_idx - (len(available_modes) - 1) / 2) * bar_w
    for model, xc in centers.items():
        df = dfs[mode]
        if df is None or model not in df.index or metric_mean not in df.columns:
            continue
        mean = df.loc[model, metric_mean]
        std = df.loc[model, metric_std] if metric_std in df.columns else 0.0
        lbl = mode_labels[mode] if mode not in added else ""
        ax.bar(xc + offset, mean, bar_w, yerr=std, color=colors[mode],
               capsize=3, alpha=0.88, label=lbl, error_kw={"linewidth": 1.2})
        added.add(mode)

for variant, label in baseline_specs:
    if variant not in baseline_dfs:
        continue
    row = baseline_dfs[variant].loc["raw_mlp"] if "raw_mlp" in baseline_dfs[variant].index else baseline_dfs[variant].iloc[0]
    mean = row[metric_mean]
    std = row[metric_std] if metric_std in row.index else 0.0
    ax.bar(baseline_x[variant], mean, bar_w * 1.2, yerr=std, capsize=3,
           color="#8C8C8C", alpha=0.88,
           label="Baselines" if variant == available_baselines[0] else "",
           error_kw={"linewidth": 1.2})

for variant, label in large_fm_specs:
    if variant not in large_fm_dfs or variant not in large_fm_x:
        continue
    row = large_fm_dfs[variant].loc["raw_mlp"] if "raw_mlp" in large_fm_dfs[variant].index else large_fm_dfs[variant].iloc[0]
    mean = row[metric_mean]
    std = row[metric_std] if metric_std in row.index else 0.0
    ax.bar(large_fm_x[variant], mean, bar_w * 1.2, yerr=std, capsize=3,
           color="#8172B3", alpha=0.88,
           label="_nolegend_",
           error_kw={"linewidth": 1.2})

for x_pos, (_name, mean, sd) in zip(ref_x, ref_models):
    ax.bar(x_pos, mean, bar_w * 1.2, yerr=sd, capsize=3,
           color="#999999", alpha=0.88, label="_nolegend_",
           error_kw={"linewidth": 1.2} if sd is not None else {})

sc_lo, sc_hi = centers["pretrain_sc"] - 0.45, centers["preadapt_sc"] + 0.45
bulk_lo, bulk_hi = centers["pretrain_bulk"] - 0.45, centers["preadapt_bulk"] + 0.45
baseline_lo = (min(baseline_x.values()) - 0.42) if baseline_x else None
baseline_hi = (max(baseline_x.values()) + 0.42) if baseline_x else None
large_fm_lo = (min(large_fm_x.values()) - 0.42) if large_fm_x else None
large_fm_hi = (max(large_fm_x.values()) + 0.42) if large_fm_x else None
ref_lo, ref_hi = ref_x[0] - 0.42, ref_x[-1] + 0.42
ax.axvspan(sc_lo, sc_hi, alpha=0.07, color="steelblue", zorder=0)
ax.axvspan(bulk_lo, bulk_hi, alpha=0.07, color="salmon", zorder=0)
if baseline_x:
    ax.axvspan(baseline_lo, baseline_hi, alpha=0.06, color="darkgray", zorder=0)
if large_fm_x:
    ax.axvspan(large_fm_lo, large_fm_hi, alpha=0.07, color="#8172B3", zorder=0)
ax.axvspan(ref_lo, ref_hi, alpha=0.05, color="gray", zorder=0)

y_lim = (0, 1.16)
ax.set_ylim(*y_lim)
ax.text((sc_lo + sc_hi) / 2, y_lim[1] - 0.015, "sc pre-trained", ha="center", va="top", fontsize=9, color="steelblue", style="italic")
ax.text((bulk_lo + bulk_hi) / 2, y_lim[1] - 0.015, "bulk pre-trained", ha="center", va="top", fontsize=9, color="firebrick", style="italic")
if baseline_x:
    ax.text((baseline_lo + baseline_hi) / 2, y_lim[1] - 0.015, "baselines", ha="center", va="top", fontsize=9, color="dimgray", style="italic")
if large_fm_x:
    ax.text((large_fm_lo + large_fm_hi) / 2, y_lim[1] - 0.015, "large FMs", ha="center", va="top", fontsize=9, color="#8172B3", style="italic")
ax.text((ref_lo + ref_hi) / 2, y_lim[1] - 0.015, "reference models", ha="center", va="top", fontsize=9, color="dimgray", style="italic")

large_fm_label_map = {variant: label for variant, label in large_fm_specs}
tick_pos = (
    list(centers.values())
    + list(baseline_x.values())
    + [large_fm_x[v] for v in available_large_fms]
    + ref_x
)
tick_labels = (
    ["Random\ninit", "Pretrain\nsc", "Preadapt\nsc", "Pretrain\nbulk", "Preadapt\nbulk"]
    + [dict(baseline_specs)[v] for v in available_baselines]
    + [large_fm_label_map[v] for v in available_large_fms]
    + [name for name, _mean, _sd in ref_models]
)
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, fontsize=8.8)
ax.set_ylabel("Weighted F1", fontsize=11)
ax.set_title("Disease Classification — Weighted F1", fontsize=12)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9.5, framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout(rect=(0, 0, 0.86, 1))
plt.show()


### Bulk pretraining dataset-size downstream comparison

In [ ]:
task = "disease_class"
base = str(OUTPUT_DIR / task)
metric_mean = "f1_weighted_mean"
metric_std = "f1_weighted_std"

bulk_downstream_specs = [
    ("10k", "head_only_pretrain_bulk_10k"),
    ("50k", "head_only_pretrain_bulk_50k"),
    ("100k", "head_only_pretrain_bulk_100k"),
    ("200k", "head_only_pretrain_bulk_200k"),
    ("400k", "head_only_pretrain_bulk_400k"),
    ("Full", None),
]


def load_bulk_downstream_result(label, variant):
    if label == "Full":
        metadata_path = f"{base}/head_only/{task}_head_only_run_metadata.json"
        candidates = [
            f"{base}/head_only/{task}_head_only_pretrain_bulk_evaluation_metrics.csv",
            f"{base}/head_only/{task}_head_only_evaluation_metrics.csv",
        ]
    else:
        prefix = f"{task}_{variant}"
        metadata_path = f"{base}/{variant}/{prefix}_run_metadata.json"
        candidates = [
            f"{base}/{variant}/{prefix}_evaluation_metrics.csv",
            f"{base}/{variant}/{prefix}_pretrain_bulk_evaluation_metrics.csv",
        ]

    if not os.path.exists(metadata_path):
        return None, None, None
    with open(metadata_path) as handle:
        metadata = __import__("json").load(handle)
    if (
        metadata.get("head_only_backbone_eval") is not True
        or metadata.get("run_status") != "complete"
    ):
        print(f"Skipping incomplete or incompatible head-only result: {label}")
        return None, None, None
    fold_fingerprint = metadata.get("cv_fold_fingerprint")
    if not fold_fingerprint:
        print(f"Skipping head-only result without fold fingerprint: {label}")
        return None, None, None

    for path in candidates:
        if not os.path.exists(path):
            continue
        frame = pd.read_csv(path, comment="#")
        if "model" in frame.columns:
            frame = frame[frame["model"] == "pretrain_bulk"]
        if not frame.empty and metric_mean in frame.columns:
            return frame.iloc[0], path, fold_fingerprint
    return None, None, None


bulk_downstream_rows = []
for label, variant in bulk_downstream_specs:
    row, path, fold_fingerprint = load_bulk_downstream_result(label, variant)
    if row is not None:
        bulk_downstream_rows.append({
            "label": label,
            "mean": float(row[metric_mean]),
            "std": float(row[metric_std]) if metric_std in row.index else 0.0,
            "path": path,
            "fold_fingerprint": fold_fingerprint,
        })

if not bulk_downstream_rows:
    raise FileNotFoundError(f"No complete bulk-size head-only results found under {base}")
if len({row["fold_fingerprint"] for row in bulk_downstream_rows}) != 1:
    raise ValueError("Bulk-size head-only results do not use identical CV folds")

x = np.arange(len(bulk_downstream_rows))
means = np.asarray([row["mean"] for row in bulk_downstream_rows])
stds = np.asarray([row["std"] for row in bulk_downstream_rows])
bar_colors = plt.get_cmap("viridis")(
    np.linspace(0.12, 0.88, len(bulk_downstream_rows))
)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(
    x, means, yerr=stds, capsize=4, width=0.68,
    color=bar_colors, alpha=0.9, error_kw={"linewidth": 1.2},
)
for bar, mean in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2, mean + 0.018, f"{mean:.3f}",
        ha="center", va="bottom", fontsize=9,
    )

ax.set_xticks(x)
ax.set_xticklabels([row["label"] for row in bulk_downstream_rows])
ax.set_ylim(0, 1.08)
ax.set_xlabel("Bulk pretraining samples", fontsize=11)
ax.set_ylabel("Weighted F1", fontsize=11)
ax.set_title("Disease classification: head-only performance by pretraining size", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


### Validation-loss convergence


In [ ]:
plot_finetuning_convergence(
    "disease_class",
    "Disease classification",
    "Validation weighted cross-entropy loss",
)


## Drug response prediction

In [ ]:
bulkformer_pcc_mean = 0.91
geneformer_pcc_mean = 0.873
genecompass_pcc_mean = 0.872
scgpt_pcc_mean = 0.877
scfoundation_pcc_mean = 0.88
sclong_pcc_mean = 0.843


In [ ]:
task = "drug_resp"
base = str(OUTPUT_DIR / task)
modes = ["head_only", "adapters", "full_ft", "pca_rf"]
mode_labels = {"head_only": "Head", "adapters": "Adapters", "full_ft": "Full", "pca_rf": "PCA+RF"}
colors = {"head_only": "#4C72B0", "adapters": "#DD8452", "full_ft": "#55A868", "pca_rf": "#8172B3"}
model_keys = ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]
metric_mean = "global_pcc_mean"
metric_std = "global_pcc_std"
baseline_specs = [
    ("raw_mlp_all_genes", "RE MLP\nAll"),
    ("raw_mlp_hvg1199", "RE MLP\nMAD"),
    ("raw_mlp_drug_only", "MLP\ndrug only"),
    ("raw_pca_rf_all_genes", "RE PCA+RF\nAll"),
    ("raw_pca_rf_hvg1199", "RE PCA+RF\nMAD"),
]
large_fm_specs = [
    ("scgpt_pca_rf", "scGPT"),
    ("scgpt_preadapt_pca_rf", "Preadapted\nscGPT"),
    ("bulkformer_pca_rf", "BulkFormer"),
]


def load_mode(mode):
    if mode == "pca_rf" and not completed_output_is_current(base, task, mode):
        print("Skipping stale or partial scbFM PCA+RF output")
        return None
    if not head_only_output_is_current(base, task, mode):
        return None
    combined = f"{base}/{mode}/{task}_{mode}_evaluation_metrics.csv"
    if os.path.exists(combined):
        return pd.read_csv(combined, comment="#").set_index("model")
    parts = [
        pd.read_csv(f"{base}/{mode}/{task}_{mode}_{m}_evaluation_metrics.csv", comment="#")
        for m in model_keys
        if os.path.exists(f"{base}/{mode}/{task}_{mode}_{m}_evaluation_metrics.csv")
    ]
    return pd.concat(parts).set_index("model") if parts else None




def load_variants(specs, require_complete=False):
    loaded = {}
    for variant, _ in specs:
        if require_complete and not completed_output_is_current(base, task, variant):
            print(f"Skipping stale or partial variant: {variant}")
            continue
        path = f"{base}/{variant}/{task}_{variant}_evaluation_metrics.csv"
        if os.path.exists(path):
            df = pd.read_csv(path, comment="#").set_index("model")
            if metric_mean in df.columns:
                loaded[variant] = df
    return loaded


dfs = {mode: load_mode(mode) for mode in modes}
available_modes = [mode for mode in modes if dfs[mode] is not None]
baseline_dfs = load_variants(baseline_specs, require_complete=True)
large_fm_dfs = load_variants(large_fm_specs, require_complete=True)
available_baselines = [variant for variant, _ in baseline_specs if variant in baseline_dfs]
available_large_fms = [variant for variant, _ in large_fm_specs if variant in large_fm_dfs]
if not available_modes and not available_baselines and not available_large_fms:
    raise FileNotFoundError(f"No evaluation metric files found under {base}")

bar_w = 0.18
centers = {"random_init": 0.0, "pretrain_sc": 1.1, "preadapt_sc": 1.9, "pretrain_bulk": 3.0, "preadapt_bulk": 3.8}
baseline_start = 5.0
baseline_spacing = 0.78
baseline_x = {variant: baseline_start + i * baseline_spacing for i, variant in enumerate(available_baselines)}
large_fm_start = (max(baseline_x.values()) + 0.95) if baseline_x else 5.0
large_fm_spacing = 0.78
large_fm_x = {variant: large_fm_start + i * large_fm_spacing for i, variant in enumerate(available_large_fms)}

ref_models = [
    ("GeneFormer",   geneformer_pcc_mean,   None),
    ("GeneCompass",  genecompass_pcc_mean,  None),
    ("scFoundation", scfoundation_pcc_mean, None),
    ("scLong",       sclong_pcc_mean,        None),
]
nonref_x = list(baseline_x.values()) + list(large_fm_x.values()) + [centers["preadapt_bulk"]]
ref_start = max(nonref_x) + 0.95
ref_spacing = 0.78
ref_x = [ref_start + i * ref_spacing for i in range(len(ref_models))]

fig, ax = plt.subplots(figsize=(18, 5.5))
added = set()
for m_idx, mode in enumerate(available_modes):
    offset = (m_idx - (len(available_modes) - 1) / 2) * bar_w
    df = dfs[mode]
    for model, xc in centers.items():
        if df is None or model not in df.index or metric_mean not in df.columns:
            continue
        mean = df.loc[model, metric_mean]
        std = df.loc[model, metric_std] if metric_std in df.columns else 0.0
        label = mode_labels[mode] if mode not in added else ""
        ax.bar(xc + offset, mean, bar_w, yerr=std, color=colors[mode], capsize=3,
               alpha=0.88, label=label, error_kw={"linewidth": 1.2})
        added.add(mode)

for variant, label in baseline_specs:
    if variant not in available_baselines:
        continue
    row = baseline_dfs[variant].loc["raw_mlp"] if "raw_mlp" in baseline_dfs[variant].index else baseline_dfs[variant].iloc[0]
    mean = row[metric_mean]
    std = row[metric_std] if metric_std in row.index else 0.0
    ax.bar(baseline_x[variant], mean, bar_w * 1.2, yerr=std, capsize=3,
           color="#8C8C8C", alpha=0.88,
           label="Baselines" if variant == available_baselines[0] else "",
           error_kw={"linewidth": 1.2})

for variant, label in large_fm_specs:
    if variant not in large_fm_dfs:
        continue
    row = large_fm_dfs[variant].iloc[0]
    mean = row[metric_mean]
    std = row[metric_std] if metric_std in row.index else 0.0
    ax.bar(large_fm_x[variant], mean, bar_w * 1.2, yerr=std, capsize=3,
           color="#8172B3", alpha=0.88, label="_nolegend_",
           error_kw={"linewidth": 1.2})

for i, (name, mean, sd) in enumerate(ref_models):
    ax.bar(ref_x[i], mean, bar_w * 1.2, yerr=sd, color="#999999",
           capsize=3, alpha=0.88, label="_nolegend_",
           error_kw={"linewidth": 1.2} if sd else {})

sc_lo, sc_hi = centers["pretrain_sc"] - 0.45, centers["preadapt_sc"] + 0.45
bulk_lo, bulk_hi = centers["pretrain_bulk"] - 0.45, centers["preadapt_bulk"] + 0.45
ax.axvspan(sc_lo, sc_hi, alpha=0.07, color="steelblue", zorder=0)
ax.axvspan(bulk_lo, bulk_hi, alpha=0.07, color="salmon", zorder=0)
y_lim = (0, 1.1)
ax.set_ylim(*y_lim)
ax.text((sc_lo + sc_hi) / 2, y_lim[1] - 0.003, "sc pre-trained", ha="center", va="top", fontsize=9, color="steelblue", style="italic")
ax.text((bulk_lo + bulk_hi) / 2, y_lim[1] - 0.003, "bulk pre-trained", ha="center", va="top", fontsize=9, color="firebrick", style="italic")
if baseline_x:
    baseline_lo, baseline_hi = min(baseline_x.values()) - 0.35, max(baseline_x.values()) + 0.35
    ax.axvspan(baseline_lo, baseline_hi, alpha=0.06, color="darkgray", zorder=0)
    ax.text((baseline_lo + baseline_hi) / 2, y_lim[1] - 0.003, "baselines", ha="center", va="top", fontsize=9, color="dimgray", style="italic")

if large_fm_x:
    large_fm_lo, large_fm_hi = min(large_fm_x.values()) - 0.35, max(large_fm_x.values()) + 0.35
    ax.axvspan(large_fm_lo, large_fm_hi, alpha=0.07, color="#8172B3", zorder=0)
    ax.text((large_fm_lo + large_fm_hi) / 2, y_lim[1] - 0.003, "large FMs", ha="center", va="top", fontsize=9, color="#8172B3", style="italic")

ref_lo, ref_hi = ref_x[0] - 0.35, ref_x[-1] + 0.35
ax.axvspan(ref_lo, ref_hi, alpha=0.05, color="gray", zorder=0)
ax.text((ref_lo + ref_hi) / 2, y_lim[1] - 0.003, "reference models", ha="center", va="top", fontsize=9, color="dimgray", style="italic")

large_fm_labels = dict(large_fm_specs)
tick_pos = list(centers.values()) + list(baseline_x.values()) + [large_fm_x[v] for v in available_large_fms] + ref_x
tick_labels = ["Random\ninit", "Pretrain\nsc", "Preadapt\nsc", "Pretrain\nbulk", "Preadapt\nbulk"] + [dict(baseline_specs)[v] for v in available_baselines] + [large_fm_labels[v] for v in available_large_fms] + [n for n, _, _ in ref_models]
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, fontsize=8.5)
ax.set_ylabel("PCC (global)", fontsize=11)
ax.set_title("Drug Response Prediction — Pearson Correlation Coefficient", fontsize=12)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9, framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout(rect=(0, 0, 0.88, 1))
plt.show()


In [ ]:
task = "drug_resp"
base = str(OUTPUT_DIR / task)
metric_mean = "global_pcc_mean"
metric_std = "global_pcc_std"
bulk_downstream_specs = [
    ("10k", "head_only_pretrain_bulk_10k"),
    ("50k", "head_only_pretrain_bulk_50k"),
    ("100k", "head_only_pretrain_bulk_100k"),
    ("200k", "head_only_pretrain_bulk_200k"),
    ("400k", "head_only_pretrain_bulk_400k"),
    ("Full", None),
]


def load_drug_bulk_size_result(label, variant):
    if label == "Full":
        metadata_path = f"{base}/head_only/{task}_head_only_run_metadata.json"
        candidates = [
            f"{base}/head_only/{task}_head_only_pretrain_bulk_evaluation_metrics.csv",
            f"{base}/head_only/{task}_head_only_evaluation_metrics.csv",
        ]
    else:
        prefix = f"{task}_{variant}"
        metadata_path = f"{base}/{variant}/{prefix}_run_metadata.json"
        candidates = [
            f"{base}/{variant}/{prefix}_evaluation_metrics.csv",
            f"{base}/{variant}/{prefix}_pretrain_bulk_evaluation_metrics.csv",
        ]
    if not os.path.exists(metadata_path):
        return None, None
    with open(metadata_path) as handle:
        metadata = json.load(handle)
    if metadata.get("head_only_backbone_eval") is not True or metadata.get("run_status") != "complete":
        print(f"Skipping incomplete or incompatible head-only result: {label}")
        return None, None
    fingerprint = metadata.get("cv_fold_fingerprint")
    if not fingerprint:
        print(f"Skipping result without fold fingerprint: {label}")
        return None, None
    for path in candidates:
        if not os.path.exists(path):
            continue
        frame = pd.read_csv(path, comment="#")
        if "model" in frame.columns:
            frame = frame[frame["model"] == "pretrain_bulk"]
        if not frame.empty and metric_mean in frame.columns:
            return frame.iloc[0], fingerprint
    return None, None


bulk_size_rows = []
for label, variant in bulk_downstream_specs:
    row, fingerprint = load_drug_bulk_size_result(label, variant)
    if row is not None:
        bulk_size_rows.append({
            "label": label,
            "mean": float(row[metric_mean]),
            "std": float(row[metric_std]) if metric_std in row.index else 0.0,
            "fold_fingerprint": fingerprint,
        })
if not bulk_size_rows:
    raise FileNotFoundError(f"No complete drug-response bulk-size results found under {base}")
if len({row["fold_fingerprint"] for row in bulk_size_rows}) != 1:
    raise ValueError("Drug-response bulk-size results do not use identical CV folds")

x = np.arange(len(bulk_size_rows))
means = np.asarray([row["mean"] for row in bulk_size_rows])
stds = np.asarray([row["std"] for row in bulk_size_rows])
bar_colors = plt.get_cmap("viridis")(np.linspace(0.12, 0.88, len(bulk_size_rows)))
fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(x, means, yerr=stds, capsize=4, width=0.68, color=bar_colors, alpha=0.9, error_kw={"linewidth": 1.2})
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.018, f"{mean:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([row["label"] for row in bulk_size_rows])
ax.set_ylim(0, 1.08)
ax.set_xlabel("Bulk pretraining samples", fontsize=11)
ax.set_ylabel("PCC (global)", fontsize=11)
ax.set_title("Drug response: head-only performance by pretraining size", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


### Validation-loss convergence


In [ ]:
plot_finetuning_convergence(
    "drug_resp",
    "Drug response prediction",
    "Validation MSE loss",
    extra_mlp_variants=[("raw_mlp_drug_only", "Drug-only MLP")],
)


## Gene essentiality prediction

In [ ]:
bulkformer_pcc_mean = 0.931
geneformer_pcc_mean = 0.897
genecompass_pcc_mean = 0.881
scgpt_pcc_mean = 0.907
scfoundation_pcc_mean = 0.852
sclong_pcc_mean = 0.889


In [ ]:
task = "gene_essent"
base = str(OUTPUT_DIR / task)
modes = ["head_only", "adapters", "full_ft", "pca_rf"]
mode_labels = {"head_only": "Head", "adapters": "Adapters", "full_ft": "Full", "pca_rf": "PCA+RF"}
colors  = {"head_only": "#4C72B0", "adapters": "#DD8452", "full_ft": "#55A868", "pca_rf": "#8172B3"}
model_keys = ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]
metric_mean = "test_pcc_mean"
metric_std = "test_pcc_std"

baseline_specs = [
    ("raw_mlp_all_genes", "RE MLP\nAll"),
    ("raw_mlp_hvg1199", "RE MLP\nMAD"),
    ("raw_pca_rf_all_genes", "RE PCA+RF\nAll"),
    ("raw_pca_rf_hvg1199", "RE PCA+RF\nMAD"),
]
large_fm_specs = [
    ("scgpt_pca_rf", "scGPT"),
    ("scgpt_preadapt_pca_rf", "Preadapted\nscGPT"),
    ("bulkformer_pca_rf", "BulkFormer"),
]

def load_eval_csv(path):
    if os.path.exists(path):
        return pd.read_csv(path, comment="#").set_index("model")
    return None


def load_mode(mode):
    if mode == "pca_rf" and not completed_output_is_current(base, task, mode):
        return None
    if not head_only_output_is_current(base, task, mode):
        return None
    combined = f"{base}/{mode}/{task}_{mode}_evaluation_metrics.csv"
    df = load_eval_csv(combined)
    if df is not None:
        return df
    parts = [
        pd.read_csv(f"{base}/{mode}/{task}_{mode}_{m}_evaluation_metrics.csv", comment="#")
        for m in model_keys
        if os.path.exists(f"{base}/{mode}/{task}_{mode}_{m}_evaluation_metrics.csv")
    ]
    return pd.concat(parts).set_index("model") if parts else None


def load_variant_metrics(specs, metric_col):
    loaded = {}
    for spec in specs:
        canonical = spec[0]
        if not completed_output_is_current(base, task, canonical):
            print(f'Skipping stale or partial baseline: {canonical}')
            continue
        path = f"{base}/{canonical}/{task}_{canonical}_evaluation_metrics.csv"
        df = load_eval_csv(path)
        if df is not None and metric_col in df.columns:
            loaded[canonical] = df
    return loaded


dfs = {mode: load_mode(mode) for mode in modes}
available_modes = [mode for mode in modes if dfs[mode] is not None]
baseline_dfs = load_variant_metrics(baseline_specs, metric_mean)
large_fm_dfs = load_variant_metrics(large_fm_specs, metric_mean)
available_baselines = [variant for variant, _ in baseline_specs if variant in baseline_dfs]
available_large_fms = [variant for variant, _label in large_fm_specs if variant in large_fm_dfs]
if not available_modes and not available_baselines and not available_large_fms:
    raise FileNotFoundError(f"No evaluation metric files found under {base}")

bar_w = 0.18
centers = {
    "random_init":  0.0,
    "pretrain_sc":  1.1,
    "preadapt_sc":  1.9,
    "pretrain_bulk": 3.0,
    "preadapt_bulk": 3.8,
}
baseline_start = 5.0
baseline_spacing = 0.78
baseline_x = {variant: baseline_start + i * baseline_spacing for i, variant in enumerate(available_baselines)}
large_fm_start = (max(baseline_x.values()) + 0.95) if baseline_x else 5.0
large_fm_spacing = 0.78
large_fm_x = {variant: large_fm_start + i * large_fm_spacing for i, variant in enumerate(available_large_fms)}
rightmost_nonref = max(
    list(baseline_x.values())
    + list(large_fm_x.values())
    + [centers["preadapt_bulk"]]
)
ref_models = [
    ("BulkFormer\n(reported)", bulkformer_pcc_mean, None),
    ("GeneFormer", geneformer_pcc_mean, None),
    ("GeneCompass", genecompass_pcc_mean, None),
    ("scGPT\n(reported)", scgpt_pcc_mean, None),
    ("scFoundation", scfoundation_pcc_mean, None),
    ("scLong", sclong_pcc_mean, None),
]
ref_start = rightmost_nonref + 1.05
ref_spacing = 0.78
ref_x = [ref_start + i * ref_spacing for i in range(len(ref_models))]

fig, ax = plt.subplots(figsize=(20, 5.5))

added = set()
for m_idx, mode in enumerate(available_modes):
    offset = (m_idx - (len(available_modes) - 1) / 2) * bar_w
    for model, xc in centers.items():
        df = dfs[mode]
        if df is None or model not in df.index or metric_mean not in df.columns:
            continue
        mean = df.loc[model, metric_mean]
        std = df.loc[model, metric_std] if metric_std in df.columns else 0.0
        lbl = mode_labels[mode] if mode not in added else ""
        ax.bar(xc + offset, mean, bar_w, yerr=std, color=colors[mode],
               capsize=3, alpha=0.88, label=lbl, error_kw={"linewidth": 1.2})
        added.add(mode)

for variant, label in baseline_specs:
    if variant not in baseline_dfs:
        continue
    row = baseline_dfs[variant].loc["raw_mlp"] if "raw_mlp" in baseline_dfs[variant].index else baseline_dfs[variant].iloc[0]
    mean = row[metric_mean]
    std = row[metric_std] if metric_std in row.index else 0.0
    ax.bar(baseline_x[variant], mean, bar_w * 1.2, yerr=std, capsize=3,
           color="#8C8C8C", alpha=0.88,
           label="Baselines" if variant == available_baselines[0] else "",
           error_kw={"linewidth": 1.2})

for variant, label in large_fm_specs:
    if variant not in large_fm_dfs or variant not in large_fm_x:
        continue
    row = large_fm_dfs[variant].loc["raw_mlp"] if "raw_mlp" in large_fm_dfs[variant].index else large_fm_dfs[variant].iloc[0]
    mean = row[metric_mean]
    std = row[metric_std] if metric_std in row.index else 0.0
    ax.bar(large_fm_x[variant], mean, bar_w * 1.2, yerr=std, capsize=3,
           color="#8172B3", alpha=0.88,
           label="_nolegend_",
           error_kw={"linewidth": 1.2})

for x_pos, (_name, mean, sd) in zip(ref_x, ref_models):
    ax.bar(x_pos, mean, bar_w * 1.2, yerr=sd, capsize=3,
           color="#999999", alpha=0.88, label="_nolegend_",
           error_kw={"linewidth": 1.2} if sd is not None else {})

sc_lo, sc_hi = centers["pretrain_sc"] - 0.45, centers["preadapt_sc"] + 0.45
bulk_lo, bulk_hi = centers["pretrain_bulk"] - 0.45, centers["preadapt_bulk"] + 0.45
baseline_lo = (min(baseline_x.values()) - 0.42) if baseline_x else None
baseline_hi = (max(baseline_x.values()) + 0.42) if baseline_x else None
large_fm_lo = (min(large_fm_x.values()) - 0.42) if large_fm_x else None
large_fm_hi = (max(large_fm_x.values()) + 0.42) if large_fm_x else None
ref_lo, ref_hi = ref_x[0] - 0.42, ref_x[-1] + 0.42
ax.axvspan(sc_lo, sc_hi, alpha=0.07, color="steelblue", zorder=0)
ax.axvspan(bulk_lo, bulk_hi, alpha=0.07, color="salmon", zorder=0)
if baseline_x:
    ax.axvspan(baseline_lo, baseline_hi, alpha=0.06, color="darkgray", zorder=0)
if large_fm_x:
    ax.axvspan(large_fm_lo, large_fm_hi, alpha=0.07, color="#8172B3", zorder=0)
ax.axvspan(ref_lo, ref_hi, alpha=0.05, color="gray", zorder=0)

y_lim = (-0.1, 1.05)
ax.set_ylim(*y_lim)
ax.text((sc_lo + sc_hi) / 2, y_lim[1] - 0.015, "sc pre-trained", ha="center", va="top", fontsize=9, color="steelblue", style="italic")
ax.text((bulk_lo + bulk_hi) / 2, y_lim[1] - 0.015, "bulk pre-trained", ha="center", va="top", fontsize=9, color="firebrick", style="italic")
if baseline_x:
    ax.text((baseline_lo + baseline_hi) / 2, y_lim[1] - 0.015, "baselines", ha="center", va="top", fontsize=9, color="dimgray", style="italic")
if large_fm_x:
    ax.text((large_fm_lo + large_fm_hi) / 2, y_lim[1] - 0.015, "large FMs", ha="center", va="top", fontsize=9, color="#8172B3", style="italic")
ax.text((ref_lo + ref_hi) / 2, y_lim[1] - 0.015, "reference models", ha="center", va="top", fontsize=9, color="dimgray", style="italic")

large_fm_label_map = {variant: label for variant, label in large_fm_specs}
tick_pos = (
    list(centers.values())
    + list(baseline_x.values())
    + [large_fm_x[v] for v in available_large_fms]
    + ref_x
)
tick_labels = (
    ["Random\ninit", "Pretrain\nsc", "Preadapt\nsc", "Pretrain\nbulk", "Preadapt\nbulk"]
    + [dict(baseline_specs)[v] for v in available_baselines]
    + [large_fm_label_map[v] for v in available_large_fms]
    + [name for name, _mean, _sd in ref_models]
)
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, fontsize=8.8)
ax.set_ylabel("Mean per-cell-line Pearson correlation", fontsize=11)
ax.set_title("Gene essentiality prediction", fontsize=12)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9.5, framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout(rect=(0, 0, 0.86, 1))
plt.show()


### Bulk pretraining dataset-size downstream comparison


In [ ]:
task = "gene_essent"
base = str(OUTPUT_DIR / task)
metric_mean = "test_pcc_mean"
metric_std = "test_pcc_std"
bulk_specs = [
    ("10k", "head_only_pretrain_bulk_10k"),
    ("50k", "head_only_pretrain_bulk_50k"),
    ("100k", "head_only_pretrain_bulk_100k"),
    ("200k", "head_only_pretrain_bulk_200k"),
    ("400k", "head_only_pretrain_bulk_400k"),
    ("Full", None),
]


def load_gene_essent_bulk_size(label, variant):
    if label == "Full":
        output_dir = "head_only"
        prefix = f"{task}_head_only"
    else:
        output_dir = variant
        prefix = f"{task}_{variant}"
    metadata_path = f"{base}/{output_dir}/{prefix}_run_metadata.json"
    metrics_path = f"{base}/{output_dir}/{prefix}_evaluation_metrics.csv"
    if not os.path.exists(metadata_path) or not os.path.exists(metrics_path):
        return None
    with open(metadata_path) as handle:
        metadata = json.load(handle)
    if (
        metadata.get("head_only_backbone_eval") is not True
        or metadata.get("run_status") != "complete"
        or not metadata.get("cv_fold_fingerprint")
        or metadata.get("gene_selection_method") != "mad"
    ):
        print(f"Skipping incomplete or incompatible result: {label}")
        return None
    frame = pd.read_csv(metrics_path, comment="#")
    frame = frame[frame["model"] == "pretrain_bulk"]
    if frame.empty or metric_mean not in frame.columns:
        return None
    row = frame.iloc[0]
    return {
        "label": label,
        "mean": float(row[metric_mean]),
        "std": float(row[metric_std]) if metric_std in row.index else 0.0,
        "fold_fingerprint": metadata["cv_fold_fingerprint"],
    }


bulk_rows = [row for label, variant in bulk_specs
             if (row := load_gene_essent_bulk_size(label, variant)) is not None]
if not bulk_rows:
    raise FileNotFoundError(f"No complete bulk-size head-only results found under {base}")
if len({row["fold_fingerprint"] for row in bulk_rows}) != 1:
    raise ValueError("Bulk-size gene-essentiality results do not use identical CV folds")

x = np.arange(len(bulk_rows))
means = np.asarray([row["mean"] for row in bulk_rows])
stds = np.asarray([row["std"] for row in bulk_rows])
bar_colors = plt.get_cmap("viridis")(np.linspace(0.12, 0.88, len(bulk_rows)))
fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(
    x, means, yerr=stds, capsize=4, width=0.68,
    color=bar_colors, alpha=0.9, error_kw={"linewidth": 1.2},
)
offset = max(float(np.max(means + stds)) * 0.025, 0.002)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, mean + offset, f"{mean:.3f}",
            ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([row["label"] for row in bulk_rows])
ax.set_ylim(min(-0.05, float(np.min(means - stds)) * 1.15), max(float(np.max(means + stds)) * 1.2, 0.2))
ax.set_xlabel("Bulk pretraining samples", fontsize=11)
ax.set_ylabel("Mean per-cell-line Pearson correlation", fontsize=11)
ax.set_title("Gene essentiality: head-only performance by pretraining size", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()



### Validation-loss convergence


In [ ]:
plot_finetuning_convergence(
    "gene_essent",
    "Gene essentiality prediction",
    "Validation MSE loss",
)


## Survival prediction

In [ ]:
bulkrnabert_weighted_c_mean = 0.642
bulkrnabert_weighted_c_sd = 0.014


In [ ]:
task = "surv_pred"
base = str(OUTPUT_DIR / task)
metric_mean, metric_std = "test_cohort_weighted_c_index_mean", "test_cohort_weighted_c_index_std"
nested_by_cancer = False
model_keys = ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]
mode_specs = [
    ("head_only", "Head", "#4C72B0"),
    ("adapters", "Adapters", "#DD8452"),
    ("full_ft", "Full", "#55A868"),
    ("pca_rf", "PCA+RF", "#8172B3"),
]
baseline_specs = [
    ("raw_mlp_all_genes", "RE MLP\nAll"),
    ("raw_mlp_hvg1199", "RE MLP\nMAD"),
    ("raw_pca_rf_all_genes", "RE PCA+RF\nAll"),
    ("raw_pca_rf_hvg1199", "RE PCA+RF\nMAD"),
]
large_fm_specs = [
    ("scgpt_pca_rf", "scGPT"),
    ("scgpt_preadapt_pca_rf", "Preadapted\nscGPT"),
    ("bulkformer_pca_rf", "BulkFormer"),
]


split_fingerprints = {}


def survival_metric_files(variant):
    prefix = f"{task}_{variant}_evaluation_metrics.csv"
    candidates = (sorted(glob.glob(f"{base}/*/{variant}/{prefix}")) if nested_by_cancer
                  else [f"{base}/{variant}/{prefix}"])
    current = []
    for path in candidates:
        metadata_path = os.path.join(os.path.dirname(path), f"{task}_{variant}_run_metadata.json")
        if not os.path.exists(path) or not os.path.exists(metadata_path):
            continue
        with open(metadata_path) as handle:
            metadata = json.load(handle)
        fingerprint_key = "survboard_split_fingerprint" if nested_by_cancer else "cv_fold_fingerprint"
        fingerprint = metadata.get(fingerprint_key)
        if metadata.get("run_status") != "complete" or not fingerprint:
            continue
        if variant == "head_only" and metadata.get("head_only_backbone_eval") is not True:
            continue
        split_key = metadata.get("cancer", "pan_cancer") if nested_by_cancer else "pan_cancer"
        previous = split_fingerprints.setdefault(split_key, fingerprint)
        if previous != fingerprint:
            raise ValueError(f"Inconsistent fold definitions for {task}/{split_key}: {variant}")
        current.append(path)
    if nested_by_cancer and current:
        expected = {"BLCA", "BRCA", "COAD", "ESCA", "HNSC", "KIRC", "KIRP", "LGG", "LUAD", "PAAD", "SARC", "SKCM", "STAD", "UCEC", "OV", "LIHC", "LUSC", "LAML", "CESC", "GBM", "READ"}
        observed = {os.path.basename(os.path.dirname(os.path.dirname(path))) for path in current}
        if observed != expected:
            print(f"Skipping incomplete {task}/{variant}: {len(observed)}/21 cohorts")
            return []
    return current


def load_survival_variant(variant):
    frames = [pd.read_csv(path, comment="#") for path in survival_metric_files(variant)]
    frames = [frame for frame in frames if metric_mean in frame.columns]
    if not frames:
        return None
    frame = pd.concat(frames, ignore_index=True)
    rows = []
    for model, group in frame.groupby("model", sort=False):
        values = pd.to_numeric(group[metric_mean], errors="coerce").dropna().to_numpy()
        if values.size == 0:
            continue
        if nested_by_cancer:
            mean, std = float(np.mean(values)), float(np.std(values, ddof=1)) if values.size > 1 else 0.0
        else:
            mean = float(values[0])
            std = float(group.iloc[0][metric_std]) if metric_std in group else 0.0
        rows.append({"model": model, "mean": mean, "std": std})
    return pd.DataFrame(rows).set_index("model") if rows else None


mode_frames = {variant: load_survival_variant(variant) for variant, _label, _color in mode_specs}
baseline_frames = {variant: load_survival_variant(variant) for variant, _label in baseline_specs}
large_frames = {variant: load_survival_variant(variant) for variant, _label in large_fm_specs}
available_modes = [variant for variant, _label, _color in mode_specs if mode_frames[variant] is not None]
available_baselines = [variant for variant, _label in baseline_specs if baseline_frames[variant] is not None]
available_large = [variant for variant, _label in large_fm_specs if large_frames[variant] is not None]
missing_variants = [variant for variant, _label, _color in mode_specs if variant not in available_modes] + [variant for variant, _label in baseline_specs if variant not in available_baselines] + [variant for variant, _label in large_fm_specs if variant not in available_large]
if missing_variants:
    print(f"{task}: not plotting missing or incomplete variants: {', '.join(missing_variants)}")
if not available_modes and not available_baselines and not available_large:
    raise FileNotFoundError(f"No complete {task} result files found under {base}")

bar_width = 0.18
centers = {"random_init": 0.0, "pretrain_sc": 1.1, "preadapt_sc": 1.9, "pretrain_bulk": 3.0, "preadapt_bulk": 3.8}
baseline_x = {variant: 5.0 + index * 0.82 for index, variant in enumerate(available_baselines)}
large_start = max(baseline_x.values(), default=4.1) + 1.0
large_x = {variant: large_start + index * 0.82 for index, variant in enumerate(available_large)}
reference_models = [("BulkRNA-Bert\n(reported)", bulkrnabert_weighted_c_mean, bulkrnabert_weighted_c_sd)]
rightmost_nonreference = max([*baseline_x.values(), *large_x.values(), centers["preadapt_bulk"]])
reference_x = [rightmost_nonreference + 1.0 + index * 0.82 for index in range(len(reference_models))]
fig, ax = plt.subplots(figsize=(17 + 0.65 * len(reference_models), 5.3))
plotted = []
for mode_index, mode in enumerate(available_modes):
    label = dict((v, l) for v, l, _c in mode_specs)[mode]
    color = dict((v, c) for v, _l, c in mode_specs)[mode]
    offset = (mode_index - (len(available_modes) - 1) / 2) * bar_width
    frame = mode_frames[mode]
    for model, center in centers.items():
        if model not in frame.index:
            continue
        row = frame.loc[model]
        ax.bar(center + offset, row["mean"], bar_width, yerr=row["std"], capsize=3,
               color=color, alpha=0.88, label=label if model == next((m for m in model_keys if m in frame.index), None) else "",
               error_kw={"linewidth": 1.2})
        plotted.append(float(row["mean"] + row["std"]))
for variant in available_baselines:
    row = baseline_frames[variant].iloc[0]
    ax.bar(baseline_x[variant], row["mean"], bar_width * 1.2, yerr=row["std"], capsize=3,
           color="#8C8C8C", alpha=0.88, error_kw={"linewidth": 1.2})
    plotted.append(float(row["mean"] + row["std"]))
for variant in available_large:
    row = large_frames[variant].iloc[0]
    ax.bar(large_x[variant], row["mean"], bar_width * 1.2, yerr=row["std"], capsize=3,
           color="#8172B3", alpha=0.88, error_kw={"linewidth": 1.2})
    plotted.append(float(row["mean"] + row["std"]))
for x_pos, (_label, mean, std) in zip(reference_x, reference_models):
    ax.bar(x_pos, mean, bar_width * 1.2, yerr=std, capsize=3,
           color="#999999", alpha=0.88, label="_nolegend_",
           error_kw={"linewidth": 1.2} if std is not None else {})
    plotted.append(float(mean + (std or 0.0)))
ax.axvspan(centers["pretrain_sc"] - 0.45, centers["preadapt_sc"] + 0.45, alpha=0.07, color="steelblue", zorder=0)
ax.axvspan(centers["pretrain_bulk"] - 0.45, centers["preadapt_bulk"] + 0.45, alpha=0.07, color="salmon", zorder=0)
if baseline_x:
    ax.axvspan(min(baseline_x.values()) - 0.42, max(baseline_x.values()) + 0.42, alpha=0.06, color="darkgray", zorder=0)
if large_x:
    ax.axvspan(min(large_x.values()) - 0.42, max(large_x.values()) + 0.42, alpha=0.07, color="#8172B3", zorder=0)
if reference_x:
    ax.axvspan(reference_x[0] - 0.42, reference_x[-1] + 0.42, alpha=0.05, color="gray", zorder=0)
label_map = dict([*(spec[:2] for spec in mode_specs), *baseline_specs, *large_fm_specs])
ticks = list(centers.values()) + list(baseline_x.values()) + list(large_x.values()) + reference_x
labels = ["Random\ninit", "Pretrain\nsc", "Preadapt\nsc", "Pretrain\nbulk", "Preadapt\nbulk"] + [label_map[v] for v in available_baselines] + [label_map[v] for v in available_large] + [label for label, _mean, _std in reference_models]
ax.set_xticks(ticks)
ax.set_xticklabels(labels, fontsize=8.5)
y_limit = max(1.05, max(plotted, default=1.0) * 1.12)
ax.set_ylim(0, y_limit)
label_y = y_limit * 0.98
ax.text(np.mean([centers["pretrain_sc"], centers["preadapt_sc"]]), label_y, "sc pre-trained", ha="center", va="top", fontsize=9, color="steelblue", style="italic")
ax.text(np.mean([centers["pretrain_bulk"], centers["preadapt_bulk"]]), label_y, "bulk pre-trained", ha="center", va="top", fontsize=9, color="firebrick", style="italic")
if baseline_x:
    ax.text(np.mean(list(baseline_x.values())), label_y, "simple baselines", ha="center", va="top", fontsize=9, color="dimgray", style="italic")
if large_x:
    ax.text(np.mean(list(large_x.values())), label_y, "large FMs", ha="center", va="top", fontsize=9, color="#8172B3", style="italic")
if reference_x:
    ax.text(np.mean(reference_x), label_y, "reference models", ha="center", va="top", fontsize=9, color="dimgray", style="italic")
ax.set_ylabel("Weighted C-index")
ax.set_title("Pan-cancer survival prediction")
ax.legend(loc="lower left", fontsize=8.5, framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

# Head-only performance by bulk pretraining dataset size. The standard full-size
# pretrain_bulk row is reused instead of rerunning an identical checkpoint.
bulk_specs = [("10k", "head_only_pretrain_bulk_10k"), ("50k", "head_only_pretrain_bulk_50k"),
              ("100k", "head_only_pretrain_bulk_100k"), ("200k", "head_only_pretrain_bulk_200k"),
              ("400k", "head_only_pretrain_bulk_400k"), ("Full", "head_only")]
bulk_rows = []
for label, variant in bulk_specs:
    frame = load_survival_variant(variant)
    if frame is None or "pretrain_bulk" not in frame.index:
        continue
    row = frame.loc["pretrain_bulk"]
    bulk_rows.append((label, float(row["mean"]), float(row["std"])))
if bulk_rows:
    x = np.arange(len(bulk_rows))
    means = np.asarray([row[1] for row in bulk_rows])
    stds = np.asarray([row[2] for row in bulk_rows])
    fig, ax = plt.subplots(figsize=(8.5, 4.7))
    bars = ax.bar(x, means, yerr=stds, capsize=4, width=0.68,
                  color=plt.get_cmap("viridis")(np.linspace(0.12, 0.88, len(bulk_rows))), alpha=0.9)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.018, f"{mean:.3f}", ha="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([row[0] for row in bulk_rows])
    ax.set_ylim(0, max(1.05, float(np.max(means + stds)) * 1.15))
    ax.set_xlabel("Bulk pretraining samples")
    ax.set_ylabel("Weighted C-index")
    ax.set_title("Pan-cancer survival prediction: head-only performance by pretraining size")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()



### Validation-loss convergence


In [ ]:
plot_finetuning_convergence(
    "surv_pred",
    "Survival prediction",
    "Validation Cox partial-likelihood loss",
)


## Survival prediction (SurvBoard)


In [ ]:
task = "surv_pred_survboard"
base = str(OUTPUT_DIR / task)
metric_mean, metric_std = "test_antolini_cindex_mean", "test_antolini_cindex_std"
nested_by_cancer = True
model_keys = ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]
mode_specs = [
    ("head_only", "Head", "#4C72B0"),
    ("adapters", "Adapters", "#DD8452"),
    ("full_ft", "Full", "#55A868"),
    ("pca_rf", "PCA+RF", "#8172B3"),
]
baseline_specs = [
    ("raw_mlp_all_genes", "RE MLP\nAll"),
    ("raw_mlp_hvg1199", "RE MLP\nMAD"),
    ("raw_pca_rf_all_genes", "RE PCA+RF\nAll"),
    ("raw_pca_rf_hvg1199", "RE PCA+RF\nMAD"),
]
large_fm_specs = [
    ("scgpt_pca_rf", "scGPT"),
    ("scgpt_preadapt_pca_rf", "Preadapted\nscGPT"),
    ("bulkformer_pca_rf", "BulkFormer"),
]


split_fingerprints = {}


def survival_metric_files(variant):
    prefix = f"{task}_{variant}_evaluation_metrics.csv"
    candidates = (sorted(glob.glob(f"{base}/*/{variant}/{prefix}")) if nested_by_cancer
                  else [f"{base}/{variant}/{prefix}"])
    current = []
    for path in candidates:
        metadata_path = os.path.join(os.path.dirname(path), f"{task}_{variant}_run_metadata.json")
        if not os.path.exists(path) or not os.path.exists(metadata_path):
            continue
        with open(metadata_path) as handle:
            metadata = json.load(handle)
        fingerprint_key = "survboard_split_fingerprint" if nested_by_cancer else "cv_fold_fingerprint"
        fingerprint = metadata.get(fingerprint_key)
        if metadata.get("run_status") != "complete" or not fingerprint:
            continue
        if variant == "head_only" and metadata.get("head_only_backbone_eval") is not True:
            continue
        split_key = metadata.get("cancer", "pan_cancer") if nested_by_cancer else "pan_cancer"
        previous = split_fingerprints.setdefault(split_key, fingerprint)
        if previous != fingerprint:
            raise ValueError(f"Inconsistent fold definitions for {task}/{split_key}: {variant}")
        current.append(path)
    if nested_by_cancer and current:
        expected = {"BLCA", "BRCA", "COAD", "ESCA", "HNSC", "KIRC", "KIRP", "LGG", "LUAD", "PAAD", "SARC", "SKCM", "STAD", "UCEC", "OV", "LIHC", "LUSC", "LAML", "CESC", "GBM", "READ"}
        observed = {os.path.basename(os.path.dirname(os.path.dirname(path))) for path in current}
        if observed != expected:
            print(f"Skipping incomplete {task}/{variant}: {len(observed)}/21 cohorts")
            return []
    return current


def load_survival_variant(variant):
    frames = [pd.read_csv(path, comment="#") for path in survival_metric_files(variant)]
    frames = [frame for frame in frames if metric_mean in frame.columns]
    if not frames:
        return None
    frame = pd.concat(frames, ignore_index=True)
    rows = []
    for model, group in frame.groupby("model", sort=False):
        values = pd.to_numeric(group[metric_mean], errors="coerce").dropna().to_numpy()
        if values.size == 0:
            continue
        if nested_by_cancer:
            mean, std = float(np.mean(values)), float(np.std(values, ddof=1)) if values.size > 1 else 0.0
        else:
            mean = float(values[0])
            std = float(group.iloc[0][metric_std]) if metric_std in group else 0.0
        rows.append({"model": model, "mean": mean, "std": std})
    return pd.DataFrame(rows).set_index("model") if rows else None


mode_frames = {variant: load_survival_variant(variant) for variant, _label, _color in mode_specs}
baseline_frames = {variant: load_survival_variant(variant) for variant, _label in baseline_specs}
large_frames = {variant: load_survival_variant(variant) for variant, _label in large_fm_specs}
available_modes = [variant for variant, _label, _color in mode_specs if mode_frames[variant] is not None]
available_baselines = [variant for variant, _label in baseline_specs if baseline_frames[variant] is not None]
available_large = [variant for variant, _label in large_fm_specs if large_frames[variant] is not None]
missing_variants = [variant for variant, _label, _color in mode_specs if variant not in available_modes] + [variant for variant, _label in baseline_specs if variant not in available_baselines] + [variant for variant, _label in large_fm_specs if variant not in available_large]
if missing_variants:
    print(f"{task}: not plotting missing or incomplete variants: {', '.join(missing_variants)}")
if not available_modes and not available_baselines and not available_large:
    raise FileNotFoundError(f"No complete {task} result files found under {base}")

bar_width = 0.18
centers = {"random_init": 0.0, "pretrain_sc": 1.1, "preadapt_sc": 1.9, "pretrain_bulk": 3.0, "preadapt_bulk": 3.8}
baseline_x = {variant: 5.0 + index * 0.82 for index, variant in enumerate(available_baselines)}
large_start = max(baseline_x.values(), default=4.1) + 1.0
large_x = {variant: large_start + index * 0.82 for index, variant in enumerate(available_large)}
reference_models = []
rightmost_nonreference = max([*baseline_x.values(), *large_x.values(), centers["preadapt_bulk"]])
reference_x = [rightmost_nonreference + 1.0 + index * 0.82 for index in range(len(reference_models))]
fig, ax = plt.subplots(figsize=(17 + 0.65 * len(reference_models), 5.3))
plotted = []
for mode_index, mode in enumerate(available_modes):
    label = dict((v, l) for v, l, _c in mode_specs)[mode]
    color = dict((v, c) for v, _l, c in mode_specs)[mode]
    offset = (mode_index - (len(available_modes) - 1) / 2) * bar_width
    frame = mode_frames[mode]
    for model, center in centers.items():
        if model not in frame.index:
            continue
        row = frame.loc[model]
        ax.bar(center + offset, row["mean"], bar_width, yerr=row["std"], capsize=3,
               color=color, alpha=0.88, label=label if model == next((m for m in model_keys if m in frame.index), None) else "",
               error_kw={"linewidth": 1.2})
        plotted.append(float(row["mean"] + row["std"]))
for variant in available_baselines:
    row = baseline_frames[variant].iloc[0]
    ax.bar(baseline_x[variant], row["mean"], bar_width * 1.2, yerr=row["std"], capsize=3,
           color="#8C8C8C", alpha=0.88, error_kw={"linewidth": 1.2})
    plotted.append(float(row["mean"] + row["std"]))
for variant in available_large:
    row = large_frames[variant].iloc[0]
    ax.bar(large_x[variant], row["mean"], bar_width * 1.2, yerr=row["std"], capsize=3,
           color="#8172B3", alpha=0.88, error_kw={"linewidth": 1.2})
    plotted.append(float(row["mean"] + row["std"]))
for x_pos, (_label, mean, std) in zip(reference_x, reference_models):
    ax.bar(x_pos, mean, bar_width * 1.2, yerr=std, capsize=3,
           color="#999999", alpha=0.88, label="_nolegend_",
           error_kw={"linewidth": 1.2} if std is not None else {})
    plotted.append(float(mean + (std or 0.0)))
ax.axvspan(centers["pretrain_sc"] - 0.45, centers["preadapt_sc"] + 0.45, alpha=0.07, color="steelblue", zorder=0)
ax.axvspan(centers["pretrain_bulk"] - 0.45, centers["preadapt_bulk"] + 0.45, alpha=0.07, color="salmon", zorder=0)
if baseline_x:
    ax.axvspan(min(baseline_x.values()) - 0.42, max(baseline_x.values()) + 0.42, alpha=0.06, color="darkgray", zorder=0)
if large_x:
    ax.axvspan(min(large_x.values()) - 0.42, max(large_x.values()) + 0.42, alpha=0.07, color="#8172B3", zorder=0)
if reference_x:
    ax.axvspan(reference_x[0] - 0.42, reference_x[-1] + 0.42, alpha=0.05, color="gray", zorder=0)
label_map = dict([*(spec[:2] for spec in mode_specs), *baseline_specs, *large_fm_specs])
ticks = list(centers.values()) + list(baseline_x.values()) + list(large_x.values()) + reference_x
labels = ["Random\ninit", "Pretrain\nsc", "Preadapt\nsc", "Pretrain\nbulk", "Preadapt\nbulk"] + [label_map[v] for v in available_baselines] + [label_map[v] for v in available_large] + [label for label, _mean, _std in reference_models]
ax.set_xticks(ticks)
ax.set_xticklabels(labels, fontsize=8.5)
y_limit = max(1.05, max(plotted, default=1.0) * 1.12)
ax.set_ylim(0, y_limit)
label_y = y_limit * 0.98
ax.text(np.mean([centers["pretrain_sc"], centers["preadapt_sc"]]), label_y, "sc pre-trained", ha="center", va="top", fontsize=9, color="steelblue", style="italic")
ax.text(np.mean([centers["pretrain_bulk"], centers["preadapt_bulk"]]), label_y, "bulk pre-trained", ha="center", va="top", fontsize=9, color="firebrick", style="italic")
if baseline_x:
    ax.text(np.mean(list(baseline_x.values())), label_y, "simple baselines", ha="center", va="top", fontsize=9, color="dimgray", style="italic")
if large_x:
    ax.text(np.mean(list(large_x.values())), label_y, "large FMs", ha="center", va="top", fontsize=9, color="#8172B3", style="italic")
if reference_x:
    ax.text(np.mean(reference_x), label_y, "reference models", ha="center", va="top", fontsize=9, color="dimgray", style="italic")
ax.set_ylabel("Antolini C-index")
ax.set_title("SurvBoard cohort-specific survival prediction")
ax.legend(loc="lower left", fontsize=8.5, framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

# Head-only performance by bulk pretraining dataset size. The standard full-size
# pretrain_bulk row is reused instead of rerunning an identical checkpoint.
bulk_specs = [("10k", "head_only_pretrain_bulk_10k"), ("50k", "head_only_pretrain_bulk_50k"),
              ("100k", "head_only_pretrain_bulk_100k"), ("200k", "head_only_pretrain_bulk_200k"),
              ("400k", "head_only_pretrain_bulk_400k"), ("Full", "head_only")]
bulk_rows = []
for label, variant in bulk_specs:
    frame = load_survival_variant(variant)
    if frame is None or "pretrain_bulk" not in frame.index:
        continue
    row = frame.loc["pretrain_bulk"]
    bulk_rows.append((label, float(row["mean"]), float(row["std"])))
if bulk_rows:
    x = np.arange(len(bulk_rows))
    means = np.asarray([row[1] for row in bulk_rows])
    stds = np.asarray([row[2] for row in bulk_rows])
    fig, ax = plt.subplots(figsize=(8.5, 4.7))
    bars = ax.bar(x, means, yerr=stds, capsize=4, width=0.68,
                  color=plt.get_cmap("viridis")(np.linspace(0.12, 0.88, len(bulk_rows))), alpha=0.9)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.018, f"{mean:.3f}", ha="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([row[0] for row in bulk_rows])
    ax.set_ylim(0, max(1.05, float(np.max(means + stds)) * 1.15))
    ax.set_xlabel("Bulk pretraining samples")
    ax.set_ylabel("Antolini C-index")
    ax.set_title("SurvBoard cohort-specific survival prediction: head-only performance by pretraining size")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()



### Validation-loss convergence


In [ ]:
plot_finetuning_convergence(
    "surv_pred_survboard",
    "SurvBoard survival prediction",
    "Validation Cox partial-likelihood loss",
    nested_by_cohort=True,
)


## Survival prediction (binary)

In [ ]:
bulkformer_auroc_mean = 0.747
geneformer_auroc_mean = 0.647
genecompass_auroc_mean = 0.709
scgpt_auroc_mean = 0.723
scfoundation_auroc_mean = 0.726
sclong_auroc_mean = 0.584


In [ ]:
task = "surv_pred_binary"
base = str(OUTPUT_DIR / task)
metric_mean, metric_std = "auroc_mean", "auroc_std"
nested_by_cancer = False
model_keys = ["random_init", "pretrain_sc", "preadapt_sc", "pretrain_bulk", "preadapt_bulk"]
mode_specs = [
    ("head_only", "Head", "#4C72B0"),
    ("adapters", "Adapters", "#DD8452"),
    ("full_ft", "Full", "#55A868"),
    ("pca_rf", "PCA+RF", "#8172B3"),
]
baseline_specs = [
    ("raw_mlp_all_genes", "RE MLP\nAll"),
    ("raw_mlp_hvg1199", "RE MLP\nMAD"),
    ("raw_pca_rf_all_genes", "RE PCA+RF\nAll"),
    ("raw_pca_rf_hvg1199", "RE PCA+RF\nMAD"),
]
large_fm_specs = [
    ("scgpt_pca_rf", "scGPT"),
    ("scgpt_preadapt_pca_rf", "Preadapted\nscGPT"),
    ("bulkformer_pca_rf", "BulkFormer"),
]


split_fingerprints = {}


def survival_metric_files(variant):
    prefix = f"{task}_{variant}_evaluation_metrics.csv"
    candidates = (sorted(glob.glob(f"{base}/*/{variant}/{prefix}")) if nested_by_cancer
                  else [f"{base}/{variant}/{prefix}"])
    current = []
    for path in candidates:
        metadata_path = os.path.join(os.path.dirname(path), f"{task}_{variant}_run_metadata.json")
        if not os.path.exists(path) or not os.path.exists(metadata_path):
            continue
        with open(metadata_path) as handle:
            metadata = json.load(handle)
        fingerprint_key = "survboard_split_fingerprint" if nested_by_cancer else "cv_fold_fingerprint"
        fingerprint = metadata.get(fingerprint_key)
        if metadata.get("run_status") != "complete" or not fingerprint:
            continue
        if variant == "head_only" and metadata.get("head_only_backbone_eval") is not True:
            continue
        split_key = metadata.get("cancer", "pan_cancer") if nested_by_cancer else "pan_cancer"
        previous = split_fingerprints.setdefault(split_key, fingerprint)
        if previous != fingerprint:
            raise ValueError(f"Inconsistent fold definitions for {task}/{split_key}: {variant}")
        current.append(path)
    if nested_by_cancer and current:
        expected = {"BLCA", "BRCA", "COAD", "ESCA", "HNSC", "KIRC", "KIRP", "LGG", "LUAD", "PAAD", "SARC", "SKCM", "STAD", "UCEC", "OV", "LIHC", "LUSC", "LAML", "CESC", "GBM", "READ"}
        observed = {os.path.basename(os.path.dirname(os.path.dirname(path))) for path in current}
        if observed != expected:
            print(f"Skipping incomplete {task}/{variant}: {len(observed)}/21 cohorts")
            return []
    return current


def load_survival_variant(variant):
    frames = [pd.read_csv(path, comment="#") for path in survival_metric_files(variant)]
    frames = [frame for frame in frames if metric_mean in frame.columns]
    if not frames:
        return None
    frame = pd.concat(frames, ignore_index=True)
    rows = []
    for model, group in frame.groupby("model", sort=False):
        values = pd.to_numeric(group[metric_mean], errors="coerce").dropna().to_numpy()
        if values.size == 0:
            continue
        if nested_by_cancer:
            mean, std = float(np.mean(values)), float(np.std(values, ddof=1)) if values.size > 1 else 0.0
        else:
            mean = float(values[0])
            std = float(group.iloc[0][metric_std]) if metric_std in group else 0.0
        rows.append({"model": model, "mean": mean, "std": std})
    return pd.DataFrame(rows).set_index("model") if rows else None


mode_frames = {variant: load_survival_variant(variant) for variant, _label, _color in mode_specs}
baseline_frames = {variant: load_survival_variant(variant) for variant, _label in baseline_specs}
large_frames = {variant: load_survival_variant(variant) for variant, _label in large_fm_specs}
available_modes = [variant for variant, _label, _color in mode_specs if mode_frames[variant] is not None]
available_baselines = [variant for variant, _label in baseline_specs if baseline_frames[variant] is not None]
available_large = [variant for variant, _label in large_fm_specs if large_frames[variant] is not None]
missing_variants = [variant for variant, _label, _color in mode_specs if variant not in available_modes] + [variant for variant, _label in baseline_specs if variant not in available_baselines] + [variant for variant, _label in large_fm_specs if variant not in available_large]
if missing_variants:
    print(f"{task}: not plotting missing or incomplete variants: {', '.join(missing_variants)}")
if not available_modes and not available_baselines and not available_large:
    raise FileNotFoundError(f"No complete {task} result files found under {base}")

bar_width = 0.18
centers = {"random_init": 0.0, "pretrain_sc": 1.1, "preadapt_sc": 1.9, "pretrain_bulk": 3.0, "preadapt_bulk": 3.8}
baseline_x = {variant: 5.0 + index * 0.82 for index, variant in enumerate(available_baselines)}
large_start = max(baseline_x.values(), default=4.1) + 1.0
large_x = {variant: large_start + index * 0.82 for index, variant in enumerate(available_large)}
reference_models = [
    ("BulkFormer\n(reported)", bulkformer_auroc_mean, None),
    ("GeneFormer", geneformer_auroc_mean, None),
    ("GeneCompass", genecompass_auroc_mean, None),
    ("scGPT\n(reported)", scgpt_auroc_mean, None),
    ("scFoundation", scfoundation_auroc_mean, None),
    ("scLong", sclong_auroc_mean, None),
]
rightmost_nonreference = max([*baseline_x.values(), *large_x.values(), centers["preadapt_bulk"]])
reference_x = [rightmost_nonreference + 1.0 + index * 0.82 for index in range(len(reference_models))]
fig, ax = plt.subplots(figsize=(17 + 0.65 * len(reference_models), 5.3))
plotted = []
for mode_index, mode in enumerate(available_modes):
    label = dict((v, l) for v, l, _c in mode_specs)[mode]
    color = dict((v, c) for v, _l, c in mode_specs)[mode]
    offset = (mode_index - (len(available_modes) - 1) / 2) * bar_width
    frame = mode_frames[mode]
    for model, center in centers.items():
        if model not in frame.index:
            continue
        row = frame.loc[model]
        ax.bar(center + offset, row["mean"], bar_width, yerr=row["std"], capsize=3,
               color=color, alpha=0.88, label=label if model == next((m for m in model_keys if m in frame.index), None) else "",
               error_kw={"linewidth": 1.2})
        plotted.append(float(row["mean"] + row["std"]))
for variant in available_baselines:
    row = baseline_frames[variant].iloc[0]
    ax.bar(baseline_x[variant], row["mean"], bar_width * 1.2, yerr=row["std"], capsize=3,
           color="#8C8C8C", alpha=0.88, error_kw={"linewidth": 1.2})
    plotted.append(float(row["mean"] + row["std"]))
for variant in available_large:
    row = large_frames[variant].iloc[0]
    ax.bar(large_x[variant], row["mean"], bar_width * 1.2, yerr=row["std"], capsize=3,
           color="#8172B3", alpha=0.88, error_kw={"linewidth": 1.2})
    plotted.append(float(row["mean"] + row["std"]))
for x_pos, (_label, mean, std) in zip(reference_x, reference_models):
    ax.bar(x_pos, mean, bar_width * 1.2, yerr=std, capsize=3,
           color="#999999", alpha=0.88, label="_nolegend_",
           error_kw={"linewidth": 1.2} if std is not None else {})
    plotted.append(float(mean + (std or 0.0)))
ax.axvspan(centers["pretrain_sc"] - 0.45, centers["preadapt_sc"] + 0.45, alpha=0.07, color="steelblue", zorder=0)
ax.axvspan(centers["pretrain_bulk"] - 0.45, centers["preadapt_bulk"] + 0.45, alpha=0.07, color="salmon", zorder=0)
if baseline_x:
    ax.axvspan(min(baseline_x.values()) - 0.42, max(baseline_x.values()) + 0.42, alpha=0.06, color="darkgray", zorder=0)
if large_x:
    ax.axvspan(min(large_x.values()) - 0.42, max(large_x.values()) + 0.42, alpha=0.07, color="#8172B3", zorder=0)
if reference_x:
    ax.axvspan(reference_x[0] - 0.42, reference_x[-1] + 0.42, alpha=0.05, color="gray", zorder=0)
label_map = dict([*(spec[:2] for spec in mode_specs), *baseline_specs, *large_fm_specs])
ticks = list(centers.values()) + list(baseline_x.values()) + list(large_x.values()) + reference_x
labels = ["Random\ninit", "Pretrain\nsc", "Preadapt\nsc", "Pretrain\nbulk", "Preadapt\nbulk"] + [label_map[v] for v in available_baselines] + [label_map[v] for v in available_large] + [label for label, _mean, _std in reference_models]
ax.set_xticks(ticks)
ax.set_xticklabels(labels, fontsize=8.5)
y_limit = max(1.05, max(plotted, default=1.0) * 1.12)
ax.set_ylim(0, y_limit)
label_y = y_limit * 0.98
ax.text(np.mean([centers["pretrain_sc"], centers["preadapt_sc"]]), label_y, "sc pre-trained", ha="center", va="top", fontsize=9, color="steelblue", style="italic")
ax.text(np.mean([centers["pretrain_bulk"], centers["preadapt_bulk"]]), label_y, "bulk pre-trained", ha="center", va="top", fontsize=9, color="firebrick", style="italic")
if baseline_x:
    ax.text(np.mean(list(baseline_x.values())), label_y, "simple baselines", ha="center", va="top", fontsize=9, color="dimgray", style="italic")
if large_x:
    ax.text(np.mean(list(large_x.values())), label_y, "large FMs", ha="center", va="top", fontsize=9, color="#8172B3", style="italic")
if reference_x:
    ax.text(np.mean(reference_x), label_y, "reference models", ha="center", va="top", fontsize=9, color="dimgray", style="italic")
ax.set_ylabel("AUROC")
ax.set_title("Binary vital-status prediction")
ax.legend(loc="lower left", fontsize=8.5, framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

# Head-only performance by bulk pretraining dataset size. The standard full-size
# pretrain_bulk row is reused instead of rerunning an identical checkpoint.
bulk_specs = [("10k", "head_only_pretrain_bulk_10k"), ("50k", "head_only_pretrain_bulk_50k"),
              ("100k", "head_only_pretrain_bulk_100k"), ("200k", "head_only_pretrain_bulk_200k"),
              ("400k", "head_only_pretrain_bulk_400k"), ("Full", "head_only")]
bulk_rows = []
for label, variant in bulk_specs:
    frame = load_survival_variant(variant)
    if frame is None or "pretrain_bulk" not in frame.index:
        continue
    row = frame.loc["pretrain_bulk"]
    bulk_rows.append((label, float(row["mean"]), float(row["std"])))
if bulk_rows:
    x = np.arange(len(bulk_rows))
    means = np.asarray([row[1] for row in bulk_rows])
    stds = np.asarray([row[2] for row in bulk_rows])
    fig, ax = plt.subplots(figsize=(8.5, 4.7))
    bars = ax.bar(x, means, yerr=stds, capsize=4, width=0.68,
                  color=plt.get_cmap("viridis")(np.linspace(0.12, 0.88, len(bulk_rows))), alpha=0.9)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.018, f"{mean:.3f}", ha="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([row[0] for row in bulk_rows])
    ax.set_ylim(0, max(1.05, float(np.max(means + stds)) * 1.15))
    ax.set_xlabel("Bulk pretraining samples")
    ax.set_ylabel("AUROC")
    ax.set_title("Binary vital-status prediction: head-only performance by pretraining size")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()



### Validation-loss convergence


In [ ]:
plot_finetuning_convergence(
    "surv_pred_binary",
    "Binary survival prediction",
    "Validation weighted cross-entropy loss",
)


## Zero-shot single-cell retention benchmarks

Frozen reference mapping and batch integration on the official scGPT COVID-19 and Lung-Kim splits.


In [ ]:
single_cell_model_order = [
    "random_init", "pretrain_sc", "preadapt_sc",
    "pretrain_bulk", "preadapt_bulk", "raw_pca",
    "scgpt", "scgpt_preadapt",
]
single_cell_positions = {
    "random_init": 0.0, "pretrain_sc": 1.1, "preadapt_sc": 1.9,
    "pretrain_bulk": 3.0, "preadapt_bulk": 3.8, "raw_pca": 5.0,
    "scgpt": 6.1, "scgpt_preadapt": 6.9,
}
single_cell_labels = {
    "random_init": "Random\ninit", "pretrain_sc": "Pretrain\nsc",
    "preadapt_sc": "Preadapt\nsc", "pretrain_bulk": "Pretrain\nbulk",
    "preadapt_bulk": "Preadapt\nbulk", "raw_pca": "Raw\nPCA",
    "scgpt": "scGPT", "scgpt_preadapt": "Preadapted\nscGPT",
}


def plot_zero_shot_single_cell(task, filename, metric, ylabel, title):
    path = os.path.expanduser(
        str(OUTPUT_DIR / task / "zero_shot_scgpt" / filename)
    )
    if not os.path.exists(path):
        print(f"Skipping {title}: missing {path}")
        return
    summary = pd.read_csv(path)
    summary = summary[summary["dataset"] == "macro_average"].set_index("model")
    models = [model for model in single_cell_model_order if model in summary.index]
    if not models:
        print(f"Skipping {title}: no macro-average rows in {path}")
        return
    colors = {model: "#4C72B0" for model in single_cell_model_order[:5]}
    colors.update({"raw_pca": "#8C8C8C", "scgpt": "#8172B3",
                   "scgpt_preadapt": "#8172B3"})
    x = np.asarray([single_cell_positions[model] for model in models])
    fig, ax = plt.subplots(figsize=(12.5, 5.3))
    ax.bar(x, summary.loc[models, metric].astype(float), width=0.58,
           color=[colors[model] for model in models], alpha=0.88)

    group_specs = [
        (("pretrain_sc", "preadapt_sc"), "sc pre-trained", "steelblue", 0.07),
        (("pretrain_bulk", "preadapt_bulk"), "bulk pre-trained", "salmon", 0.07),
        (("raw_pca",), "raw baseline", "darkgray", 0.06),
        (("scgpt", "scgpt_preadapt"), "external scGPT", "#8172B3", 0.07),
    ]
    y_max = 1.16
    for group_models, group_label, color, alpha in group_specs:
        group_x = [single_cell_positions[model] for model in group_models if model in models]
        if not group_x:
            continue
        lower, upper = min(group_x) - 0.43, max(group_x) + 0.43
        ax.axvspan(lower, upper, color=color, alpha=alpha, zorder=0)
        text_color = {"bulk pre-trained": "firebrick",
                      "raw baseline": "dimgray"}.get(group_label, color)
        ax.text((lower + upper) / 2, y_max - 0.015, group_label,
                ha="center", va="top", fontsize=9, color=text_color, style="italic")

    ax.set_xticks(x, [single_cell_labels[model] for model in models], fontsize=8.8)
    ax.set_ylim(0, y_max)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.spines[["top", "right"]].set_visible(False)
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor="#4C72B0", alpha=0.88, label="scbFM"),
        Patch(facecolor="#8C8C8C", alpha=0.88, label="Raw PCA"),
        Patch(facecolor="#8172B3", alpha=0.88, label="scGPT"),
    ], loc="upper left", bbox_to_anchor=(1.01, 1.0), framealpha=0.9)
    fig.tight_layout(rect=(0, 0, 0.86, 1))
    plt.show()


plot_zero_shot_single_cell(
    "cell_type_annotation", "cell_type_annotation_summary.csv",
    "macro_f1", "Macro F1", "Zero-shot cell-type annotation: Macro F1",
)
plot_zero_shot_single_cell(
    "batch_integration", "batch_integration_summary.csv",
    "overall", "Overall integration score",
    "Zero-shot batch integration: overall score",
)


## Combined downstream convergence figure

One row per downstream task and one column per neural fine-tuning regime.

In [ ]:
from pathlib import Path
import importlib
import sys
import matplotlib.pyplot as plt

analysis_dir = REPO_ROOT / "src/analysis"
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import downstream_convergence
downstream_convergence = importlib.reload(downstream_convergence)

combined_convergence_figure = downstream_convergence.plot_downstream_convergence(
    output_root=OUTPUT_DIR,
    figure_dir=FIGURE_DIR,
)
plt.show()
